# DB7-025: reference-capacity

**Question:** Does finer reference resolution improve development accuracy?

Compare 2x2x2=8, 3x2x2=12, 3x3x2=18 and 3x3x3=27 rules. Three-reference axes use 0, training-fold median clipped to [0.1,0.9], and 1. Keep the chosen inputs/consequents unchanged. Equal accuracy favours fewer rules.

## Fixed protocol and assumptions

- Exercise B, E1 labels 1–17, all 22 subjects; rest excluded using restimulus/rerepetition.
- Within-subject neural experts: train repetitions 1/3/4, development repetition 6, test repetitions 2/5. This deliberately differs from the earlier four-training-repetition final model.
- 200 ms (400 samples at 2 kHz), 10 ms (20 samples) stride; windows stay within an active repetition. No new boundary trim is added.
- EMG 12 channels, fourth-order 20–450 Hz Butterworth plus 50 Hz notch (Q=30), inherited zero-phase filtering per repetition. ACC, gyroscope and magnetometer each retain 36 channels and original offsets; align/resample per repetition. Input z-scaling and spectral scaling use neural training only.
- The unchanged W/S/I experts use the existing temporal CNN architecture. Each branch is an independent 17-class expert, not a newly trained joint fusion network. Fixed 13 epochs, dropout .65, batch 512, Adam, learning rates 1e-3/1e-4/1e-5 in epochs 1–3/4–9/10–13; augmentation off.
- A compact shared gate is fitted across subjects. Five outer subject-disjoint development folds evaluate each candidate; three inner subject folds produce baseline outputs for gate training. Neural experts remain within-subject. This is not LOSO neural evaluation.
- Fit meta models on up to 32 evenly spaced windows per subject/gesture/trial, with equal trial weights; evaluate all windows. These are correlated samples, not 32 independent trials.
- Select mean development subject accuracy, then fewer rules, then prespecified order. Preserve the global baseline whenever no candidate beats it. No test-based tuning. Later notebooks read only the previous selected configuration.
- Test inference is cached before gate selection for efficiency, but the meta fitting/selection code does not load test labels until writing the selection lock. This is reused DB7 data, not an untouched confirmatory dataset.

## BRB inputs, rules and outputs

All antecedents lie in [0,1]. With three two-reference inputs, the Cartesian product yields eight rules. Matching is piecewise linear; rule activation is the normalized product of matching degrees, optionally raised to learned positive attribute weights and multiplied by rule weights. Consequent belief degrees are softmax-normalized and sum to one. ER combines complete beliefs; Sugeno uses the same rule activations and weighted consequent average. Logistic is a matched non-rule control.

Correctness heads output P(incorrect), P(correct) for each expert. Fusion multiplies the development-fitted global expert weights by predicted correctness and renormalizes. Recovery heads output P(recovery), P(harm), P(neither); they compare an observable challenger against the global baseline. If recovery > harm, choose the challenger, otherwise retain the baseline. Initial beliefs are smoothed training outcome frequencies. SciPy L-BFGS-B fits consequent logits with analytic gradients. Test labels never fit rules.

## Required outputs and interpretation

Export every development candidate and optimization diagnostic; selection lock; exact checkpoint provenance; initial and trained rules; reference values, rule/attribute weights; fitted models; per-window predictions; per-subject/gesture/repetition/phase errors; recovery/harm counts; confusion matrices; accuracy, macro F1 and balanced accuracy; genuine probability NLL/Brier/ECE where defined. Export paired subject bootstrap intervals, subject sign-flip and Wilcoxon tests. Window-level exact McNemar is descriptive only because windows overlap 95%. Multiple arms and sequential research decisions make inferential results exploratory. No method is promised a 90% chance of improvement.

The selected-policy and structural-candidate results are both retained. A structural candidate can lose while the selected policy correctly remains the baseline. Compare with this study's matched-checkpoint baseline, not directly with the older four-repetition accuracy.


In [ ]:
import os
for name in ['OMP_NUM_THREADS','OPENBLAS_NUM_THREADS','MKL_NUM_THREADS']:
    os.environ[name]='1'
from pathlib import Path
SOURCE=Path('/kaggle/working/db7_suite_source')
SOURCE.mkdir(exist_ok=True)


## three_branch_model.py

Frozen neural architecture: waveform, spectral and inertial encoders and temporal classifier.

In [ ]:
%%writefile /kaggle/working/db7_suite_source/three_branch_model.py
"""C1: frame-aligned EMG waveform, log-power and inertial encoders for DB7.

The caller supplies filtered, training-channel-standardized windows in this order:
EMG (12), ACC (36), gyroscope (36), magnetometer (36). This module does not read
recordings, fit the raw input scaler, choose data splits, or filter signals.

Before training, call ``model.fit_spectral_scaler(training_loader, split='train')``.
That loader must contain only already-standardized training windows. Spectral
statistics are registered buffers and travel with every model checkpoint.
"""

from __future__ import annotations

import io as checkpoint_io
from collections.abc import Iterable
from typing import Any

import torch
from torch import Tensor, nn
from torch.nn import functional as F


EXPECTED_PARAMETERS = 551_542
EXPECTED_PARAMETER_BREAKDOWN = {
    'waveform': 81_492,
    'spectral': 67_936,
    'inertial': 123_760,
    'fusion': 41_216,
    'temporal': 197_888,
    'attention': 4_161,
    'classifier': 35_089,
}
FRAME_SAMPLES = 200
FRAME_HOP = 100
WINDOW_SAMPLES = 400
N_FRAMES = 3


def unfold_frames(signal: Tensor) -> Tensor:
    """Return [batch, channels, 3, 200], using only the supplied 400 rows."""
    if signal.ndim != 3 or signal.shape[-1] != WINDOW_SAMPLES:
        raise ValueError(f'Expected [batch, channels, 400], received {tuple(signal.shape)}')
    return signal.unfold(-1, FRAME_SAMPLES, FRAME_HOP)


def log_power(z_emg: Tensor, hann: Tensor | None = None) -> Tensor:
    """Return unscaled log-power [batch, 12, 44, 3] for 20:10:450 Hz.

    ``z_emg`` has already received the fixed training input scaler. Explicit
    200-sample frames avoid the FFT-length-dependent framing of torch.stft.
    The FFT is at least float32, including inside an autocast context: CUDA
    half-precision FFTs cannot implement this non-power-of-two length.
    """
    if z_emg.ndim != 3 or z_emg.shape[1:] != (12, WINDOW_SAMPLES):
        raise ValueError(f'Expected [batch, 12, 400], received {tuple(z_emg.shape)}')
    if not z_emg.is_floating_point():
        raise TypeError('EMG input must be a floating-point tensor')
    fft_dtype = torch.float64 if z_emg.dtype == torch.float64 else torch.float32
    if hann is None:
        hann = torch.hann_window(FRAME_SAMPLES, periodic=True,
                                 device=z_emg.device, dtype=fft_dtype)
    else:
        hann = hann.to(device=z_emg.device, dtype=fft_dtype)
        if hann.shape != (FRAME_SAMPLES,):
            raise ValueError('Hann window must contain exactly 200 samples')
    with torch.autocast(device_type=z_emg.device.type, enabled=False):
        frames = unfold_frames(z_emg.to(dtype=fft_dtype))
        spectrum = torch.fft.rfft(frames * hann, n=FRAME_SAMPLES, dim=-1)
        power = spectrum.abs().square() / hann.square().sum()
        # [B, 12, frame, frequency] -> [B, 12, frequency, frame].
        return torch.log(power[..., 2:46] + 1e-8).permute(0, 1, 3, 2).contiguous()


def _conv_bn_relu(in_channels: int, out_channels: int, kernel: int) -> nn.Sequential:
    return nn.Sequential(
        nn.Conv1d(in_channels, out_channels, kernel, padding=kernel // 2, bias=False),
        nn.BatchNorm1d(out_channels, eps=1e-5, momentum=0.1),
        nn.ReLU(),
    )


class ChannelSqueezeExcitation(nn.Module):
    def __init__(self, channels: int) -> None:
        super().__init__()
        self.gate = nn.Sequential(
            nn.Linear(channels, channels // 8), nn.ReLU(),
            nn.Linear(channels // 8, channels), nn.Sigmoid(),
        )

    def forward(self, x: Tensor) -> Tensor:
        return x * self.gate(x.mean(-1)).unsqueeze(-1)


class FrameMultiKernelBlock(nn.Module):
    """Three single-convolution paths, plus projected residual and frame SE."""
    def __init__(self, in_channels: int, out_channels: int) -> None:
        super().__init__()
        self.paths = nn.ModuleList(_conv_bn_relu(in_channels, 32, k) for k in (3, 5, 7))
        self.merge = nn.Sequential(
            nn.Conv1d(96, out_channels, 1, bias=False),
            nn.BatchNorm1d(out_channels, eps=1e-5, momentum=0.1),
        )
        self.skip = nn.Conv1d(in_channels, out_channels, 1, bias=False)
        self.se = ChannelSqueezeExcitation(out_channels)

    def forward(self, x: Tensor) -> Tensor:
        merged = self.merge(torch.cat([path(x) for path in self.paths], dim=1))
        return self.se(F.relu(merged + self.skip(x)))


class WaveformEncoder(nn.Module):
    def __init__(self) -> None:
        super().__init__()
        self.stages = nn.Sequential(
            FrameMultiKernelBlock(12, 64), nn.MaxPool1d(2),
            FrameMultiKernelBlock(64, 96), nn.MaxPool1d(2),
        )
        self.projection = nn.Linear(192, 96)

    def forward(self, frames: Tensor) -> Tensor:
        # The same encoder processes every frame; batch and frame remain distinct.
        batch = frames.shape[0]
        x = frames.permute(0, 2, 1, 3).reshape(batch * N_FRAMES, 12, FRAME_SAMPLES)
        x = self.stages(x)
        x = F.relu(self.projection(torch.cat([x.mean(-1), x.amax(-1)], dim=1)))
        return x.reshape(batch, N_FRAMES, 96).transpose(1, 2).contiguous()


class SpectralEncoder(nn.Module):
    def __init__(self) -> None:
        super().__init__()
        layers: list[nn.Module] = []
        for incoming, outgoing, kernel, stride in ((12, 32, 5, 2),
                                                   (32, 64, 5, 2),
                                                   (64, 96, 3, 1)):
            layers.extend([
                nn.Conv2d(incoming, outgoing, (kernel, 1), stride=(stride, 1),
                          padding=(kernel // 2, 0), bias=False),
                nn.BatchNorm2d(outgoing, eps=1e-5, momentum=0.1), nn.ReLU(),
            ])
        self.stages = nn.Sequential(*layers)
        self.projection = nn.Conv1d(384, 96, 1, bias=True)

    def forward(self, x: Tensor) -> Tensor:
        x = self.stages(x)
        # Preserve four ordered feature-frequency regions, rather than collapsing
        # absolute frequency location into a single global mean/max vector.
        regions = torch.stack([x[:, :, begin:end, :].mean(2)
                               for begin, end in ((0, 3), (3, 6), (6, 9), (9, 11))], dim=2)
        # Channel-major: each channel retains regions low -> high in adjacent slots.
        return F.relu(self.projection(regions.flatten(1, 2)))


class InertialModalityEncoder(nn.Module):
    def __init__(self) -> None:
        super().__init__()
        self.stages = nn.Sequential(
            _conv_bn_relu(36, 48, 5), nn.AvgPool1d(4),
            _conv_bn_relu(48, 64, 5),
        )
        self.projection = nn.Linear(128, 48)

    def forward(self, frames: Tensor) -> Tensor:
        x = self.stages(frames)
        return F.relu(self.projection(torch.cat([x.mean(-1), x.amax(-1)], dim=1)))


class InertialEncoder(nn.Module):
    def __init__(self) -> None:
        super().__init__()
        self.modalities = nn.ModuleList(InertialModalityEncoder() for _ in range(3))
        self.dynamic_projection = nn.Linear(144, 128)
        self.mean_projection = nn.Linear(108, 128)

    def forward(self, frames: Tensor) -> Tensor:
        batch = frames.shape[0]
        x = frames.permute(0, 2, 1, 3).reshape(batch * N_FRAMES, 108, FRAME_SAMPLES)
        encoded = [encoder(x[:, i * 36:(i + 1) * 36, :])
                   for i, encoder in enumerate(self.modalities)]
        dynamic = self.dynamic_projection(torch.cat(encoded, dim=1))
        static = self.mean_projection(x.mean(-1))
        # The mean is an additional feature; it is never subtracted from frames.
        output = F.relu(dynamic + static)
        return output.reshape(batch, N_FRAMES, 128).transpose(1, 2).contiguous()


class ResidualTemporalBlock(nn.Module):
    def __init__(self, dilation: int, dropout: float) -> None:
        super().__init__()
        self.layers = nn.Sequential(
            nn.Conv1d(128, 192, 3, dilation=dilation, padding=dilation, bias=False),
            nn.BatchNorm1d(192, eps=1e-5, momentum=0.1), nn.ReLU(), nn.Dropout(dropout),
            nn.Conv1d(192, 128, 1, bias=False),
            nn.BatchNorm1d(128, eps=1e-5, momentum=0.1), nn.Dropout(dropout),
        )

    def forward(self, x: Tensor) -> Tensor:
        return F.relu(x + self.layers(x))


class ThreeBranchC1(nn.Module):
    """Exact 551,542-parameter C1, compatible with Trainer's model(x) call."""
    def __init__(self, dropout: float = 0.15) -> None:
        super().__init__()
        self.waveform = WaveformEncoder()
        self.spectral = SpectralEncoder()
        self.inertial = InertialEncoder()
        self.fusion = nn.Sequential(
            nn.Conv1d(320, 128, 1, bias=False),
            nn.BatchNorm1d(128, eps=1e-5, momentum=0.1), nn.ReLU(),
        )
        self.temporal = nn.Sequential(ResidualTemporalBlock(1, dropout),
                                      ResidualTemporalBlock(2, dropout))
        self.attention = nn.Sequential(nn.Linear(128, 32), nn.Tanh(), nn.Linear(32, 1))
        self.classifier = nn.Sequential(nn.Linear(256, 128), nn.ReLU(),
                                        nn.Dropout(dropout), nn.Linear(128, 17))
        self.register_buffer('hann', torch.hann_window(FRAME_SAMPLES, periodic=True))
        self.register_buffer('spectral_mean', torch.zeros(12, 44))
        self.register_buffer('spectral_std', torch.ones(12, 44))
        self.register_buffer('spectral_constant', torch.zeros(12, 44, dtype=torch.bool))
        self.register_buffer('spectral_fitted', torch.tensor(False))
        self.register_buffer('spectral_fit_frames', torch.tensor(0, dtype=torch.long))
        if self.count_params() != EXPECTED_PARAMETERS:
            raise AssertionError(f'C1 parameter mismatch: {self.count_params()}')

    def count_params(self) -> int:
        return sum(parameter.numel() for parameter in self.parameters())

    def parameter_breakdown(self) -> dict[str, int]:
        return {name: sum(parameter.numel() for parameter in getattr(self, name).parameters())
                for name in EXPECTED_PARAMETER_BREAKDOWN}

    @torch.no_grad()
    def fit_spectral_scaler(self, training_batches: Iterable[Any], *, split: str = 'train',
                            std_floor: float = 1e-6) -> dict[str, Any]:
        """Fit frozen moments without running a CNN or updating BatchNorm.

        The iterable yields standardized x or (x,y) batches. One observation for
        each channel/frequency bin is one frame; all three frames of every
        training window receive equal weight. Overlap is intentional. Population
        variance uses a stable float64 batched merge; SD below ``std_floor`` is
        flagged and clamped. A second fit is rejected to avoid accidental reuse
        on validation/test; create a fresh model for a new subject or fold.

        The explicit split guard cannot establish arbitrary generator provenance.
        The caller must save source/window hashes. A DataLoader exposing a dataset
        with a ``meta['split']`` column receives an additional provenance check.
        """
        if split != 'train':
            raise ValueError('Spectral scaler may be fitted only on the train split')
        if bool(self.spectral_fitted.item()):
            raise RuntimeError('Spectral scaler is already fitted; use a fresh model for another fold')
        if std_floor <= 0:
            raise ValueError('std_floor must be positive')
        dataset = getattr(training_batches, 'dataset', None)
        metadata = getattr(dataset, 'meta', None)
        if metadata is not None and 'split' in metadata:
            if set(metadata['split'].unique()) != {'train'}:
                raise ValueError('Spectral fitting DataLoader contains non-training metadata')
        device = self.spectral_mean.device
        count = 0
        mean = torch.zeros(12, 44, dtype=torch.float64, device=device)
        m2 = torch.zeros_like(mean)
        for batch in training_batches:
            x = batch[0] if isinstance(batch, (tuple, list)) else batch
            x = torch.as_tensor(x, device=device, dtype=self.spectral_mean.dtype)
            if x.ndim != 3 or x.shape[1:] != (120, WINDOW_SAMPLES) or x.shape[0] == 0:
                raise ValueError('Spectral scaler requires nonempty [B,120,400] training batches')
            if not bool(torch.isfinite(x).all().item()):
                raise ValueError('Nonfinite training input in spectral scaler fit')
            values = log_power(x[:, :12], self.hann).to(torch.float64)
            batch_count = values.shape[0] * N_FRAMES
            batch_mean = values.mean(dim=(0, 3))
            batch_m2 = (values - batch_mean[None, :, :, None]).square().sum(dim=(0, 3))
            combined_count = count + batch_count
            delta = batch_mean - mean
            m2 += batch_m2 + delta.square() * (count * batch_count / combined_count)
            mean += delta * (batch_count / combined_count)
            count = combined_count
        if not count:
            raise ValueError('Cannot fit the spectral scaler on an empty iterable')
        std = (m2 / count).clamp_min(0).sqrt()
        self.spectral_mean.copy_(mean)
        self.spectral_constant.copy_(std < std_floor)
        self.spectral_std.copy_(std.clamp_min(std_floor))
        self.spectral_fit_frames.fill_(count)
        self.spectral_fitted.fill_(True)
        return {
            'split': 'train', 'windows': count // N_FRAMES, 'frames': count,
            'statistic': 'float64_population_moments_over_training_windows_and_three_frames',
            'std_floor': float(std_floor),
            'constant_bins': int(self.spectral_constant.sum().item()),
            'mean': self.spectral_mean.detach().cpu().tolist(),
            'std': self.spectral_std.detach().cpu().tolist(),
            'constant': self.spectral_constant.detach().cpu().tolist(),
        }

    def forward_features(self, x: Tensor, *, retain_sequences: bool = False) -> dict[str, Any]:
        if x.ndim != 3 or x.shape[1:] != (120, WINDOW_SAMPLES):
            raise ValueError(f'C1 expects [B,120,400], received {tuple(x.shape)}')
        if not bool(self.spectral_fitted.item()):
            raise RuntimeError('Fit the spectral scaler using training windows before model(x)')
        frames = unfold_frames(x)
        waveform = self.waveform(frames[:, :12])
        power = log_power(x[:, :12], self.hann)
        scaled_power = ((power - self.spectral_mean[None, :, :, None]) /
                        self.spectral_std[None, :, :, None])
        spectral = self.spectral(scaled_power)
        inertial = self.inertial(frames[:, 12:])
        fused = self.temporal(self.fusion(torch.cat([waveform, spectral, inertial], dim=1)))
        scores = self.attention(fused.transpose(1, 2)).squeeze(-1)
        attention = torch.softmax(scores, dim=-1)
        weighted = (fused * attention[:, None, :]).sum(-1)
        embedding = torch.cat([weighted, fused.mean(-1)], dim=1)
        result = {
            'logits': self.classifier(embedding), 'embedding': embedding,
            'branch_embeddings': {'waveform': waveform.mean(-1),
                                  'spectral': spectral.mean(-1),
                                  'inertial': inertial.mean(-1)},
            'attention': attention,
        }
        if retain_sequences:
            result['sequences'] = {'waveform': waveform, 'spectral': spectral,
                                   'inertial': inertial, 'fused': fused}
        return result

    def forward(self, x: Tensor) -> Tensor:
        return self.forward_features(x)['logits']


def model_preflight(device: str | torch.device = 'cpu') -> dict[str, Any]:
    """Meaningful synthetic checks for the installed Kaggle PyTorch runtime.

    Does not touch recordings or disk, install packages, or start an experiment.
    CPU and applicable CUDA RNG state are restored before return.
    """
    target = torch.device(device)
    # torch.manual_seed also seeds CUDA generators. Preserve every initialized
    # device's stream instead of changing an unused second Kaggle GPU's RNG.
    cuda_devices = list(range(torch.cuda.device_count())) if torch.cuda.is_available() else []
    with torch.random.fork_rng(devices=cuda_devices):
        torch.manual_seed(941)
        model = ThreeBranchC1().to(target)
        assert model.parameter_breakdown() == EXPECTED_PARAMETER_BREAKDOWN
        x = torch.randn(3, 120, WINDOW_SAMPLES, device=target)
        try:
            model(x)
        except RuntimeError as error:
            assert 'spectral scaler' in str(error)
        else:
            raise AssertionError('Unfitted spectral scaling must block model inference')
        try:
            model.fit_spectral_scaler([x], split='test')
        except ValueError as error:
            assert 'train split' in str(error)
        else:
            raise AssertionError('The spectral scaler must reject non-training splits')
        frames = unfold_frames(x)
        assert frames.shape == (3, 120, 3, 200)
        for frame in range(N_FRAMES):
            torch.testing.assert_close(frames[:, :, frame], x[:, :, frame * 100:frame * 100 + 200])
        # Independent explicit-frame transform verifies frequency/frame ordering.
        hann = torch.hann_window(200, periodic=True, device=target)
        reference = torch.stack([
            torch.log(torch.fft.rfft(x[:, :12, begin:begin + 200] * hann, dim=-1)
                      .abs().square()[..., 2:46] / hann.square().sum() + 1e-8)
            for begin in range(0, 201, 100)], dim=-1)
        torch.testing.assert_close(log_power(x[:, :12]), reference)
        before_bn = {name: value.clone() for name, value in model.named_buffers()
                     if 'running_' in name or 'num_batches_tracked' in name}
        statistics = model.fit_spectral_scaler([(x[:2], torch.zeros(2)),
                                                (x[2:], torch.zeros(1))], split='train')
        assert statistics['windows'] == 3 and statistics['frames'] == 9
        try:
            model.fit_spectral_scaler([x], split='train')
        except RuntimeError as error:
            assert 'already fitted' in str(error)
        else:
            raise AssertionError('A fitted spectral scaler must reject accidental refitting')
        reference_double = reference.double()
        expected_mean = reference_double.mean(dim=(0, 3))
        expected_std = reference_double.permute(1, 2, 0, 3).reshape(12, 44, -1).std(-1, correction=0)
        torch.testing.assert_close(model.spectral_mean, expected_mean.float(), rtol=2e-5, atol=2e-6)
        torch.testing.assert_close(model.spectral_std, expected_std.float(), rtol=2e-5, atol=2e-6)
        for name, value in model.named_buffers():
            if name in before_bn:
                torch.testing.assert_close(value, before_bn[name], rtol=0, atol=0)
        frozen_scaler = {name: value.clone() for name, value in model.named_buffers()
                         if name.startswith('spectral_')}
        model.train()
        outputs = model.forward_features(x, retain_sequences=True)
        assert outputs['logits'].shape == (3, 17)
        assert outputs['embedding'].shape == (3, 256)
        assert outputs['attention'].shape == (3, 3)
        for branch, width in (('waveform', 96), ('spectral', 96), ('inertial', 128), ('fused', 128)):
            assert outputs['sequences'][branch].shape == (3, width, 3)
        torch.testing.assert_close(outputs['attention'].sum(-1), torch.ones(3, device=target))
        loss = F.cross_entropy(outputs['logits'], torch.tensor([0, 8, 16], device=target))
        loss.backward()
        for name, module in (('waveform', model.waveform), ('spectral', model.spectral),
                             ('acc', model.inertial.modalities[0]),
                             ('gyro', model.inertial.modalities[1]),
                             ('mag', model.inertial.modalities[2]),
                             ('inertial_mean', model.inertial.mean_projection),
                             ('fusion', model.fusion)):
            gradients = [p.grad for p in module.parameters() if p.grad is not None]
            assert gradients and all(bool(torch.isfinite(g).all().item()) for g in gradients), name
            assert any(bool(g.abs().sum().item() > 0) for g in gradients), name
        for name, value in model.named_buffers():
            if name in frozen_scaler:
                torch.testing.assert_close(value, frozen_scaler[name], rtol=0, atol=0)
        model.eval()
        with torch.no_grad():
            expected = model(x)
            # An input to another branch cannot change waveform/spectral tokens.
            changed = x.clone()
            changed[:, 12:] += 1.75
            original_features = model.forward_features(x, retain_sequences=True)
            altered_features = model.forward_features(changed, retain_sequences=True)
            for name in ('waveform', 'spectral'):
                torch.testing.assert_close(original_features['sequences'][name],
                                           altered_features['sequences'][name], rtol=0, atol=0)
            checkpoint = checkpoint_io.BytesIO()
            torch.save(model.state_dict(), checkpoint)
            checkpoint.seek(0)
            restored = ThreeBranchC1().to(target)
            restored.load_state_dict(torch.load(checkpoint, map_location=target, weights_only=True))
            restored.eval()
            torch.testing.assert_close(restored(x), expected, rtol=0, atol=0)
        return {'success': True, 'parameters': model.count_params(),
                'parameter_breakdown': model.parameter_breakdown(), 'device': str(target),
                'torch_version': torch.__version__, 'frames': 3, 'frequency_bins': 44,
                'output_shape': list(expected.shape),
                'checks': ['exact_parameter_count', 'frame_alignment', 'explicit_fft_reference',
                           'unfitted_nontraining_and_refit_guards',
                           'training_spectral_moments', 'scaler_fit_does_not_update_batchnorm',
                           'frozen_scaler', 'forward_backward_all_branches',
                           'branch_input_isolation', 'checkpoint_round_trip']}


if __name__ == '__main__':
    import json
    print(json.dumps(model_preflight('cuda' if torch.cuda.is_available() else 'cpu'), indent=2))


## ablation_model.py

Frozen independent W, S and I expert definitions, preserving the original parameters and forward computation.

In [ ]:
%%writefile /kaggle/working/db7_suite_source/ablation_model.py
"""Retrained branch subsets. Full WSI is numerically identical to original C1."""
import torch
from torch import nn
from three_branch_model import ThreeBranchC1, unfold_frames, log_power

ARMS=('W','S','I','WS','WI','SI','WSI')
NAMES={'W':'waveform','S':'spectral','I':'inertial'}
SLICES={'W':(0,96),'S':(96,192),'I':(192,320)}

class AblationC1(ThreeBranchC1):
    def __init__(self,arm,dropout=.15):
        assert arm in ARMS
        super().__init__(dropout)
        self.arm=arm
        self.active_names=[NAMES[k] for k in arm]
        indices=[i for k in arm for i in range(*SLICES[k])]
        if arm!='WSI':
            old=self.fusion[0]
            # Preserve the shared initialization; restore RNG after constructor.
            with torch.random.fork_rng():
                replacement=nn.Conv1d(len(indices),128,1,bias=False)
            with torch.no_grad():replacement.weight.copy_(old.weight[:,indices])
            self.fusion[0]=replacement
            for k,name in NAMES.items():
                if k not in arm:delattr(self,name)

    def forward_features(self,x,retain_sequences=False):
        frames=unfold_frames(x);seq={}
        if 'W' in self.arm:seq['waveform']=self.waveform(frames[:,:12])
        if 'S' in self.arm:
            if not self.spectral_fitted:raise RuntimeError('Fit spectral scaler first')
            p=log_power(x[:,:12],self.hann)
            seq['spectral']=self.spectral((p-self.spectral_mean[None,:,:,None])/self.spectral_std[None,:,:,None])
        if 'I' in self.arm:seq['inertial']=self.inertial(frames[:,12:])
        fused=self.temporal(self.fusion(torch.cat(list(seq.values()),dim=1)))
        attention=self.attention(fused.transpose(1,2)).squeeze(-1).softmax(-1)
        embedding=torch.cat([(fused*attention[:,None]).sum(-1),fused.mean(-1)],dim=1)
        out=dict(logits=self.classifier(embedding),embedding=embedding,
                 attention=attention,branch_embeddings={k:v.mean(-1) for k,v in seq.items()})
        if retain_sequences:out['sequences']={**seq,'fused':fused}
        return out

def preflight():
    """Exact full-model parity; gradient coverage and excluded-input isolation."""
    x=torch.randn(3,120,400,device='cuda')
    torch.manual_seed(77);ref=ThreeBranchC1().cuda()
    ref.fit_spectral_scaler([x]);ref.eval()
    torch.manual_seed(77);full=AblationC1('WSI').cuda()
    full.load_state_dict(ref.state_dict());full.eval()
    torch.testing.assert_close(full(x),ref(x),rtol=0,atol=0)
    rows=[]
    for arm in ARMS:
        model=AblationC1(arm).cuda()
        if 'S' in arm:model.fit_spectral_scaler([x])
        model.train();out=model(x)
        nn.functional.cross_entropy(out,torch.tensor([0,8,16],device='cuda')).backward()
        assert all(p.grad is not None and torch.isfinite(p.grad).all() for p in model.parameters())
        model.eval()
        changed=x.clone()
        if 'I' not in arm:changed[:,12:]+=10
        if arm=='I':changed[:,:12]+=10
        torch.testing.assert_close(model(x),model(changed),rtol=0,atol=0)
        rows.append(dict(arm=arm,parameters=model.count_params()))
    return rows


## tc_support.py

Frozen data loader and preprocessing. Its convenience four-repetition scaler is ignored; suite_neural explicitly fits and verifies the three-repetition scaler.

In [ ]:
%%writefile /kaggle/working/db7_suite_source/tc_support.py
"""C1 under TC-AiFusion's split and training budget, retaining all inertial sensors.

No TC feature images or external-window history are used: these would replace C1.
All preprocessing fits use training repetitions only. Test is evaluated once,
after the fixed final epoch. This is annotation-assisted offline classification.
"""
import gc, hashlib, json, random, time, traceback, zipfile
from datetime import datetime, timezone
from pathlib import Path
import numpy as np
import pandas as pd
from scipy.io import loadmat
from scipy.signal import butter, sosfiltfilt, iirnotch, filtfilt, resample_poly
import torch
from torch.nn import functional as F
from torch.utils.data import Dataset, DataLoader
from three_branch_model import ThreeBranchC1, model_preflight


class Settings:
    SUBJECTS = list(range(1, 21))
    TRAIN_REPS = [1, 3, 4, 6]
    TEST_REPS = [2, 5]
    FS = 2000
    WINDOW = 400  # 200 ms
    STEP = 20     # 10 ms
    BATCH_SIZE = 512
    EPOCHS = 13
    DROPOUT = 0.65  # Match TC-AiFusion's training regularization setting.
    SEED = 42
    SMOKE = False
    INPUT = None
    OUTPUT = Path('/kaggle/working') if Path('/kaggle').exists() else Path.cwd()
    AUTOMATION = {}


def write_json(path, value):
    Path(path).parent.mkdir(parents=True, exist_ok=True)
    Path(path).write_text(json.dumps(value,indent=2),encoding='utf-8')


def find_input():
    if Settings.INPUT is not None:return Path(Settings.INPUT)
    for candidate in sorted(Path('/kaggle/input').rglob('Subject_1')):
        if candidate.is_dir():return candidate.parent
    raise FileNotFoundError('Attach rayaanraza1/ninapro-db7 or set Settings.INPUT to the subject-folder root.')


def aligned_interval(array, emg_length, start, end):
    """TC alignment rule, independently applied to ACC, gyro and magnetometer."""
    if len(array)==emg_length:return array[start:end].astype(np.float32,copy=True)
    ratio=len(array)/float(emg_length)
    a0=max(0,min(int(np.floor(start*ratio)),len(array)-1))
    a1=max(a0+1,min(int(np.ceil(end*ratio)),len(array)))
    segment=array[a0:a1];length=end-start
    divisor=np.gcd(len(segment),length)
    result=resample_poly(segment,length//divisor,len(segment)//divisor,axis=0).astype(np.float32)
    if len(result)<length:result=np.vstack([result,np.repeat(result[-1:],length-len(result),axis=0)])
    return result[:length]


def load_subject(root, subject, output):
    directory=root/f'Subject_{subject}'
    if not directory.exists():directory=root/f'S{subject}'
    files=sorted(directory.rglob('*_E1_*.mat'))
    if len(files)!=1:raise ValueError(f'S{subject}: expected one E1 file, found {len(files)}')
    data=loadmat(files[0],variable_names=['emg','acc','gyro','mag','restimulus','rerepetition','subject','exercise'])
    def sensor(key,channels):
        x=np.asarray(data[key],dtype=np.float32)
        if x.ndim!=2:raise ValueError(f'{key}: not 2D')
        if x.shape[1]!=channels and x.shape[0]==channels:x=x.T
        if x.shape[1]!=channels or not np.isfinite(x).all():raise ValueError(f'{key}: invalid channels/values')
        return x
    emg=sensor('emg',12)
    inertial=[sensor(key,36) for key in ['acc','gyro','mag']]
    labels=np.asarray(data['restimulus']).reshape(-1).astype(int)
    reps=np.asarray(data['rerepetition']).reshape(-1).astype(int)
    assert len(emg)==len(labels)==len(reps), 'Reject silent label/signal truncation'
    assert set(np.unique(labels))==set(range(18))
    edges=np.r_[0,np.flatnonzero(np.diff(labels))+1,len(labels)]
    sos=butter(4,[20,450],btype='bandpass',fs=Settings.FS,output='sos')
    b,a=iirnotch(50,30,fs=Settings.FS)
    segments=[];records=[]
    for start,end in zip(edges[:-1],edges[1:]):
        gesture=int(labels[start])
        if gesture==0:continue
        native=np.unique(reps[start:end])
        if len(native)!=1 or native[0] not in range(1,7):
            raise ValueError('Gesture run must contain exactly one valid repetition; do not assign by majority')
        repetition=int(native[0]);split='train' if repetition in Settings.TRAIN_REPS else 'test'
        filtered=sosfiltfilt(sos,emg[start:end],axis=0)
        filtered=filtfilt(b,a,filtered,axis=0).astype(np.float32)
        x=np.concatenate([filtered]+[aligned_interval(v,len(emg),int(start),int(end)) for v in inertial],axis=1)
        if not np.isfinite(x).all():raise ValueError('Nonfinite filtered/aligned signal')
        segments.append(x)
        records.append(dict(subject=subject,gesture=gesture,native_repetition=repetition,split=split,
            run_start=int(start),run_end=int(end),file_name=files[0].name,segment=len(segments)-1))
    inventory=pd.DataFrame(records)
    for gesture in range(1,18):
        rows=inventory[inventory.gesture==gesture]
        assert len(rows)==6 and set(rows.native_repetition)==set(range(1,7))
    # TC normalization: every sample of each active training segment counted once.
    total=np.zeros(120,np.float64);squares=total.copy();count=0
    for rec in records:
        if rec['split']=='train':
            x=segments[rec['segment']].astype(np.float64)
            total+=x.sum(0);squares+=np.square(x).sum(0);count+=len(x)
    mean=total/count;variance=np.maximum(squares/count-mean*mean,1e-12)
    std=np.sqrt(variance)
    np.savez_compressed(output/'input_scaler.npz',mean=mean.astype(np.float32),std=std.astype(np.float32),training_sample_count=count)
    inventory.to_csv(output/'repetition_inventory.csv',index=False)
    write_json(output/'identity.json',dict(folder_subject=subject,internal_subject=int(np.asarray(data.get('subject',subject)).item()),
        file=files[0].name,emg_sha256=hashlib.sha256(emg.tobytes()).hexdigest(),
        sensor_shapes={key:list(value.shape) for key,value in zip(['emg','acc','gyro','mag'],[emg]+inertial)},
        alignment_fallback={key:len(value)!=len(emg) for key,value in zip(['acc','gyro','mag'],inertial)}))
    return segments,inventory,mean.astype(np.float32),std.astype(np.float32)


class TCWindows(Dataset):
    def __init__(self,segments,inventory,mean,std,split):
        self.segments=segments;self.mean=mean;self.std=std
        rows=[]
        for rec in inventory[inventory.split==split].to_dict('records'):
            for offset in range(0,rec['run_end']-rec['run_start']-Settings.WINDOW+1,Settings.STEP):
                start=rec['run_start']+offset;end=start+Settings.WINDOW
                phase=((start+end)/2-rec['run_start'])/(rec['run_end']-rec['run_start'])
                rows.append(dict(**rec,offset=offset,window_start=start,window_end=end,
                    phase_fraction=phase,phase='early' if phase<1/3 else 'middle' if phase<2/3 else 'late'))
        self.meta=pd.DataFrame(rows)
        self.y=self.meta.gesture.to_numpy()-1
        self.locations=self.meta[['segment','offset']].to_numpy()
        assert not self.meta.duplicated(['subject','gesture','native_repetition','window_start']).any()
    def __len__(self):return len(self.y)
    def __getitem__(self,index):
        seg,start=self.locations[index]
        x=self.segments[seg][start:start+Settings.WINDOW]
        return torch.from_numpy(((x-self.mean)/self.std).T.copy()),int(self.y[index])


def set_seed(seed):
    random.seed(seed);np.random.seed(seed);torch.manual_seed(seed)
    if torch.cuda.is_available():torch.cuda.manual_seed_all(seed)


def evaluate(model,dataset,device,path):
    path.mkdir(exist_ok=True)
    model.eval();probs=[];embeddings=[];attention=[]
    branch={key:[] for key in model.active_names}
    with torch.no_grad():
        for x,y in DataLoader(dataset,batch_size=Settings.BATCH_SIZE,shuffle=False):
            out=model.forward_features(x.to(device))
            probs.append(out['logits'].softmax(1).cpu().numpy());embeddings.append(out['embedding'].cpu().numpy())
            attention.append(out['attention'].cpu().numpy())
            for key in branch:branch[key].append(out['branch_embeddings'][key].cpu().numpy())
    p=np.concatenate(probs);pred=p.argmax(1)
    assert np.isfinite(p).all() and np.allclose(p.sum(1),1,atol=1e-5)
    frame=dataset.meta.copy();frame['seed_base']=Settings.SEED;frame['arm']=model.arm;frame['prediction']=pred+1;frame['correct']=pred==dataset.y;frame['confidence']=p.max(1)
    if path.name == 'test':
        frame.to_csv(path/'predictions.csv',index=False)
        arrays=dict(y_true=dataset.y, probabilities=p)
        if Settings.SEED==42:
            arrays.update(embeddings=np.concatenate(embeddings),attention=np.concatenate(attention),
                          **{k:np.concatenate(v) for k,v in branch.items()})
        np.savez_compressed(path/'probabilities_embeddings.npz',**arrays)
    errors=frame.groupby(['subject','gesture','native_repetition']).agg(windows=('correct','size'),correct=('correct','sum')).reset_index()
    errors['wrong']=errors.windows-errors.correct;errors['error_percent']=100*errors.wrong/errors.windows
    errors.to_csv(path/'gesture_errors.csv',index=False)
    phase=frame.groupby(['gesture','native_repetition','phase']).agg(windows=('correct','size'),correct=('correct','sum')).reset_index()
    phase['wrong']=phase.windows-phase.correct;phase.to_csv(path/'phase_errors.csv',index=False)
    confusion=pd.crosstab(frame.gesture,frame.prediction).reindex(index=range(1,18),columns=range(1,18),fill_value=0)
    confusion.to_csv(path/'confusion.csv')
    tp=np.diag(confusion);den=confusion.sum(axis=0).to_numpy()+confusion.sum(axis=1).to_numpy()
    f1=2*tp/np.maximum(den,1)
    metrics=dict(accuracy=float(frame.correct.mean()),macro_f1=float(f1.mean()),n_windows=len(frame),
        wrong=int((~frame.correct).sum()),nll=float(-np.log(p[np.arange(len(p)),dataset.y].clip(1e-12)).mean()))
    write_json(path/'metrics.json',metrics)
    return metrics




## brb_worker.py

Frozen neural fitting, physical descriptors, hashes and GPU preflight helpers. The historical main routine is not invoked.

In [ ]:
%%writefile /kaggle/working/db7_suite_source/brb_worker.py
"""Isolated-GPU neural/OOF worker for the DB7 BRB pilot.

Each subject/seed has 12 leave-one-training-repetition-out expert fits and five
final fits. Calibration and reliability learning finish before test inference.
The unchanged TC loader supplies physical segments; its convenience all-four
input scaler is explicitly ignored and every fold computes its own statistics.
"""
from __future__ import annotations

import argparse
import gc
import hashlib
import importlib
import json
import os
from pathlib import Path
import time
import traceback

import numpy as np
import pandas as pd

TRAIN_REPS = (1, 3, 4, 6)
TEST_REPS = (2, 5)
EXPERTS = ("W", "S", "I")
FINAL_ARMS = (*EXPERTS, "SI", "WSI")
KEYS = ["subject", "gesture", "native_repetition", "window_start", "window_end"]


def json_write(path, value):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    temporary = path.with_name(path.name + ".tmp")
    temporary.write_text(json.dumps(value, indent=2, allow_nan=False), encoding="utf-8")
    temporary.replace(path)


def digest(value):
    return hashlib.sha256(json.dumps(value, sort_keys=True, separators=(",", ":"),
                                     allow_nan=False).encode()).hexdigest()


def file_hash(path):
    result = hashlib.sha256()
    with Path(path).open("rb") as handle:
        for block in iter(lambda: handle.read(1024 * 1024), b""):
            result.update(block)
    return result.hexdigest()


def window_hash(meta):
    return hashlib.sha256(meta[KEYS].to_csv(index=False).encode()).hexdigest()


def training_signal_hash(segments, metadata):
    """Bind checkpoints to physical training samples, not just their moments."""
    if set(metadata["split"]) != {"train"} or not set(metadata.native_repetition).issubset(TRAIN_REPS):
        raise ValueError("Training signal hash requires development-only train metadata")
    result = hashlib.sha256()
    for segment in sorted(map(int, metadata.segment.unique())):
        values = np.ascontiguousarray(segments[segment])
        result.update(json.dumps([segment, list(values.shape), str(values.dtype)]).encode())
        result.update(memoryview(values).cast("B"))
    return result.hexdigest()


def atomic_npz(path, **arrays):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    temporary = path.with_name(path.name + ".tmp")
    with temporary.open("wb") as handle:
        np.savez_compressed(handle, **arrays)
    temporary.replace(path)


def validate_config(config):
    """Fail rather than silently modifying the accepted pilot protocol."""
    fixed = dict(window_samples=400, stride_samples=20, fs=2000, batch_size=512,
                 dropout=0.65, epochs=2 if config.get("smoke", False) else 13)
    for key, value in fixed.items():
        if config.get(key, value) != value:
            raise ValueError(f"Unexpected {key}: this pilot requires {value}")
    if tuple(config.get("train_repetitions", TRAIN_REPS)) != TRAIN_REPS:
        raise ValueError("Development repetitions must be 1/3/4/6")
    if tuple(config.get("test_repetitions", TEST_REPS)) != TEST_REPS:
        raise ValueError("Outer test repetitions must be 2/5")
    if config.get("reference_cap_per_trial", 128) < 1:
        raise ValueError("Reference cap must be positive")
    if not config.get("subjects") or not config.get("seeds"):
        raise ValueError("Explicit subjects and seeds are required")
    if config.get("smoke", False):
        if config.get("smoke_windows_per_trial", 32) < 1:
            raise ValueError("Smoke windows per trial must be positive")
        if config.get("smoke_max_train_batches", 2) < 1:
            raise ValueError("Smoke training-batch cap must be positive")


def fold_inventory(inventory, training_repetitions, evaluation_repetitions):
    """Copy the inventory and assign splits solely from native repetition IDs."""
    train = set(map(int, training_repetitions))
    evaluation = set(map(int, evaluation_repetitions))
    if not train or not train.issubset(TRAIN_REPS):
        raise ValueError("Only development repetitions can train a model")
    if not evaluation or train.intersection(evaluation):
        raise ValueError("Training and evaluation repetitions must be disjoint")
    if not evaluation.issubset(set(TRAIN_REPS) | set(TEST_REPS)):
        raise ValueError("Unknown evaluation repetition")
    result = inventory.copy()
    result["split"] = "excluded"
    result.loc[result.native_repetition.isin(train), "split"] = "train"
    result.loc[result.native_repetition.isin(evaluation), "split"] = "evaluation"
    for split in ("train", "evaluation"):
        selected = result.loc[result.split == split]
        if set(selected.gesture) != set(range(1, 18)):
            raise ValueError(f"{split}: all 17 gestures are required")
        if selected.duplicated(["subject", "gesture", "native_repetition"]).any():
            raise ValueError("Expected one segment per subject/gesture/repetition")
        expected = train if split == "train" else evaluation
        for _, group in selected.groupby("gesture"):
            if set(group.native_repetition) != expected:
                raise ValueError("Gesture repetition coverage is incomplete")
    return result


def fit_input_scaler(segments, inventory, training_repetitions):
    """Every physical training sample counts once, independent of window overlap."""
    expected = set(map(int, training_repetitions))
    rows = inventory.loc[inventory.split == "train"]
    if not expected or not expected.issubset(TRAIN_REPS):
        raise ValueError("Scaler training repetitions must be development-only")
    if set(rows.native_repetition) != expected:
        raise ValueError("Scaler inventory provenance does not match this fold")
    total = np.zeros(120, dtype=np.float64)
    m2 = np.zeros(120, dtype=np.float64)
    count = 0
    for row in rows.to_dict("records"):
        values = np.asarray(segments[int(row["segment"])] , dtype=np.float64)
        if values.ndim != 2 or values.shape[1] != 120 or not np.isfinite(values).all():
            raise ValueError("Invalid physical training segment")
        batch_count = len(values)
        if batch_count == 0:
            raise ValueError("Empty training segment")
        batch_mean = values.mean(axis=0)
        combined = count + batch_count
        delta = batch_mean - total
        m2 += ((values - batch_mean) ** 2).sum(axis=0) + delta ** 2 * count * batch_count / combined
        total += delta * batch_count / combined
        count = combined
    if not count:
        raise ValueError("Empty scaler training population")
    std = np.sqrt(np.maximum(m2 / count, 1e-12))
    provenance = dict(training_repetitions=sorted(expected), training_sample_count=count,
                      training_segments=len(rows), statistic="population_sample_moments",
                      variance_floor=1e-12,
                      inventory_sha256=hashlib.sha256(rows.to_csv(index=False).encode()).hexdigest())
    return total.astype(np.float32), std.astype(np.float32), provenance


def balanced_indices(metadata, cap):
    """Use equal deterministic, time-spread window counts for every trial."""
    groups = list(metadata.groupby(["subject", "gesture", "native_repetition"], sort=True).indices.values())
    if not groups or cap < 1:
        raise ValueError("Nonempty metadata and positive cap are required")
    count = min(int(cap), min(map(len, groups)))
    return np.sort(np.concatenate([indices[np.linspace(0, len(indices) - 1, count).round().astype(int)]
                                   for indices in groups]))


def make_windows(segments, inventory, mean, std, split, smoke=False, smoke_cap=32):
    support = importlib.import_module("tc_support")
    dataset = support.TCWindows(segments, inventory, mean, std, split)
    if not len(dataset):
        raise ValueError("No windows were constructed")
    if smoke:
        keep = balanced_indices(dataset.meta, smoke_cap)
        dataset.meta = dataset.meta.iloc[keep].reset_index(drop=True)
        dataset.y = dataset.y[keep]
        dataset.locations = dataset.locations[keep]
    if set(dataset.y) != set(range(17)):
        raise ValueError("All 17 classes must occur even in smoke mode")
    return dataset


def raw_descriptors(windows):
    """Physical features; no learned transform and no labels used here.

    Input [N,120,400], output RMS12, Hann-FFT mean-frequency12, inertial mean108.
    The feature FFT deliberately covers the entire 200 ms outer window and is
    separate from the model's 100 ms frame FFT. Frequencies include 20..450 Hz.
    """
    values = np.asarray(windows, dtype=np.float64)
    if values.ndim != 3 or values.shape[1:] != (120, 400) or not np.isfinite(values).all():
        raise ValueError("Expected finite physical windows [N,120,400]")
    emg = values[:, :12]
    rms = np.sqrt(np.square(emg).mean(axis=-1))
    frequencies = np.fft.rfftfreq(400, d=1 / 2000)
    selected = (frequencies >= 20) & (frequencies <= 450)
    power = np.abs(np.fft.rfft(emg * np.hanning(400), axis=-1)) ** 2
    power = power[:, :, selected]
    total = power.sum(axis=-1)
    mean_frequency = np.divide((power * frequencies[selected]).sum(axis=-1), total,
                               out=np.zeros_like(total), where=total > 0)
    means = values[:, 12:].mean(axis=-1)
    return np.concatenate([rms, mean_frequency, means], axis=1)


def dataset_descriptors(dataset):
    arrays = []
    for begin in range(0, len(dataset), 128):
        values = np.stack([dataset.segments[int(segment)][int(offset):int(offset) + 400].T
                           for segment, offset in dataset.locations[begin:begin + 128]])
        arrays.append(raw_descriptors(values))
    result = np.concatenate(arrays)
    if result.shape != (len(dataset), 132):
        raise ValueError("Physical feature dimensionality mismatch")
    return result


def feature_columns():
    return ([f"emg_{channel:02}_rms" for channel in range(1, 13)] +
            [f"emg_{channel:02}_mean_frequency" for channel in range(1, 13)] +
            [f"{sensor}_{channel:02}_mean" for sensor in ("acc", "gyro", "mag")
             for channel in range(1, 37)])


def fit_shift_reference(raw_features, metadata, training_repetitions, cap=128):
    """Fit log floor, robust centers and scales using only this fold's train trials."""
    raw_features = np.asarray(raw_features, dtype=np.float64)
    if raw_features.shape != (len(metadata), 132) or not np.isfinite(raw_features).all():
        raise ValueError("Invalid training physical features")
    expected = set(map(int, training_repetitions))
    if not expected.issubset(TRAIN_REPS) or set(metadata.native_repetition) != expected:
        raise ValueError("Reference features contain unexpected or test repetitions")
    if set(metadata["split"]) != {"train"}:
        raise ValueError("Reference feature metadata must be explicitly training")
    keep = balanced_indices(metadata, cap)
    selected = raw_features[keep]
    # Every floor is fitted from the same capped, equally weighted training trials.
    rms_floor = np.maximum(selected[:, :12].max(axis=0) * 1e-6, 1e-12)
    transformed = selected.copy()
    transformed[:, :12] = np.log(np.maximum(transformed[:, :12], rms_floor))
    center = np.median(transformed, axis=0)
    q25, q75 = np.percentile(transformed, [25, 75], axis=0)
    iqr = q75 - q25
    scale_floor = np.maximum(np.abs(transformed).max(axis=0) * 1e-6, 1e-12)
    scale = np.maximum(iqr, scale_floor)
    reference = dict(version=1, training_repetitions=sorted(expected),
                     available_training_windows=len(metadata), selected_training_windows=len(keep),
                     trials=int(metadata.groupby(["subject", "gesture", "native_repetition"]).ngroups),
                     requested_cap_per_trial=int(cap), effective_windows_per_trial=len(keep) //
                     int(metadata.groupby(["subject", "gesture", "native_repetition"]).ngroups),
                     selected_window_sha256=window_hash(metadata.iloc[keep]),
                     source_feature_sha256=hashlib.sha256(np.ascontiguousarray(selected).tobytes()).hexdigest(),
                     feature_columns=feature_columns(), rms_floor=rms_floor.tolist(),
                     center=center.tolist(), scale=scale.tolist(), iqr=iqr.tolist(),
                     scale_floor=scale_floor.tolist(),
                     formula="mean(abs((feature-center)/scale))/(1+mean(abs((feature-center)/scale)))",
                     waveform_transform="natural_log(max(physical_rms, training_rms_floor))",
                     spectral_descriptor="mean_frequency: symmetric_Hann_FFT400_20_to_450Hz",
                     arm_feature_slices={"W": [0, 12], "S": [12, 24], "I": [24, 132]})
    reference["sha256"] = digest(reference)
    return reference, keep


def apply_shift_reference(raw_features, reference):
    values = np.asarray(raw_features, dtype=np.float64).copy()
    if values.ndim != 2 or values.shape[1] != 132 or not np.isfinite(values).all():
        raise ValueError("Invalid physical feature array")
    values[:, :12] = np.log(np.maximum(values[:, :12], reference["rms_floor"]))
    deviations = np.abs((values - reference["center"]) / reference["scale"])
    shifts = np.column_stack([deviations[:, begin:end].mean(axis=1)
                              for begin, end in ((0, 12), (12, 24), (24, 132))])
    shifts = shifts / (1 + shifts)
    if not np.isfinite(shifts).all() or np.any(shifts < 0) or np.any(shifts > 1):
        raise ValueError("Invalid normalized shift")
    return shifts.astype(np.float32)


def probability_metrics(probabilities, y):
    p = np.asarray(probabilities, dtype=np.float64)
    y = np.asarray(y, dtype=int)
    if p.shape != (len(y), 17) or not np.isfinite(p).all() or np.any(p < 0):
        raise ValueError("Invalid 17-class probability output")
    if not np.allclose(p.sum(axis=1), 1, atol=1e-5) or not len(y):
        raise ValueError("Invalid probability normalization or empty targets")
    if y.min() < 0 or y.max() > 16:
        raise ValueError("Expected zero-based gesture targets")
    predicted = p.argmax(axis=1)
    correct = predicted == y
    confusion = np.zeros((17, 17), dtype=np.int64)
    np.add.at(confusion, (y, predicted), 1)
    f1 = 2 * confusion.diagonal() / np.maximum(confusion.sum(0) + confusion.sum(1), 1)
    recall = confusion.diagonal() / np.maximum(confusion.sum(1), 1)
    onehot = np.eye(17)[y]
    confidence = p.max(axis=1)
    ece = 0.0
    for lower in np.arange(15) / 15:
        selection = (confidence >= lower) & ((confidence < lower + 1 / 15) if lower < 14 / 15 else (confidence <= 1))
        if selection.any():
            ece += selection.mean() * abs(confidence[selection].mean() - correct[selection].mean())
    return dict(n_windows=len(y), correct=int(correct.sum()), wrong=int((~correct).sum()),
                accuracy=float(correct.mean()), macro_f1=float(f1.mean()),
                balanced_accuracy=float(recall.mean()),
                nll=float(-np.log(p[np.arange(len(y)), y].clip(1e-12)).mean()),
                brier=float(np.square(p - onehot).sum(axis=1).mean()), ece15=float(ece))


def save_predictions(folder, dataset, logits):
    logits = np.asarray(logits, dtype=np.float32)
    if logits.shape != (len(dataset), 17) or not np.isfinite(logits).all():
        raise ValueError("Invalid logits")
    shifted = logits.astype(np.float64) - logits.max(axis=1, keepdims=True)
    probs = np.exp(shifted)
    probs /= probs.sum(axis=1, keepdims=True)
    folder = Path(folder)
    folder.mkdir(parents=True, exist_ok=True)
    atomic_npz(folder / "predictions.npz", logits=logits, probabilities=probs.astype(np.float32), y=dataset.y)
    frame = dataset.meta.copy()
    frame["prediction"] = probs.argmax(axis=1) + 1
    frame["correct"] = frame.prediction.to_numpy() == frame.gesture.to_numpy()
    frame["confidence"] = probs.max(axis=1)
    frame.to_csv(folder / "predictions.csv", index=False)
    json_write(folder / "metrics.json", probability_metrics(probs, dataset.y))
    return probs.astype(np.float32)


def model_predict(model, dataset, batch_size):
    import torch
    from torch.utils.data import DataLoader
    model.eval()
    pieces = []
    with torch.no_grad():
        for x, _ in DataLoader(dataset, batch_size=batch_size, shuffle=False, num_workers=0):
            pieces.append(model(x.cuda()).cpu().numpy())
    return np.concatenate(pieces)


def train_fit(arm, seed, subject, train, folder, fold_tag, run_signature, config):
    """Return the final fixed-epoch model; exact completed fits may be reused."""
    import torch
    from torch.nn import functional as F
    from torch.utils.data import DataLoader
    from ablation_model import AblationC1
    import tc_support as support
    folder = Path(folder)
    folder.mkdir(parents=True, exist_ok=True)
    smoke = bool(config.get("smoke", False))
    epochs = 2 if smoke else 13
    actual_seed = int(seed) + 1000 * int(subject)
    fit_spec = dict(run_signature=run_signature, subject=int(subject), seed=int(seed),
                    actual_seed=actual_seed, arm=arm, fold=fold_tag, epochs=epochs,
                    stage="oof" if fold_tag.startswith("oof_") else "final",
                    held_out_repetition=int(fold_tag.rsplit("_", 1)[1]) if fold_tag.startswith("oof_") else None,
                    training_window_sha256=window_hash(train.meta),
                    training_signal_sha256=training_signal_hash(train.segments, train.meta),
                    input_scaler_sha256=hashlib.sha256(train.mean.tobytes() + train.std.tobytes()).hexdigest(),
                    training_repetitions=sorted(map(int, train.meta.native_repetition.unique())),
                    training_windows=len(train), smoke=smoke)
    signature = digest(fit_spec)
    manifest_path = folder / "fit_manifest.json"
    checkpoint = folder / "final_model.pt"
    if manifest_path.exists():
        prior = json.loads(manifest_path.read_text(encoding="utf-8"))
        if prior.get("signature") != signature:
            raise RuntimeError(f"Refusing incompatible fit reuse: {folder}")
        if prior.get("success"):
            if not checkpoint.exists() or file_hash(checkpoint) != prior.get("checkpoint_sha256"):
                raise RuntimeError("Completed checkpoint missing or corrupted")
            model = AblationC1(arm, config.get("dropout", 0.65)).cuda()
            model.load_state_dict(torch.load(checkpoint, map_location="cuda", weights_only=True))
            print(f"RESUME verified {folder}", flush=True)
            return model
    json_write(manifest_path, dict(**fit_spec, signature=signature, success=False,
                                  state="training", status="training", epochs_run=0,
                                  metric_usage="no evaluation checkpoint selection"))
    support.set_seed(actual_seed)
    model = AblationC1(arm, config.get("dropout", 0.65)).cuda()
    batch_size = int(config.get("batch_size", 512))
    if "S" in arm:
        scaler_info = model.fit_spectral_scaler(DataLoader(train, batch_size=batch_size, shuffle=False,
                                                           num_workers=0), split="train")
        scaler_info.update(training_window_sha256=fit_spec["training_window_sha256"],
                           training_repetitions=fit_spec["training_repetitions"], fold=fold_tag)
        json_write(folder / "spectral_scaler.json", scaler_info)
    support.set_seed(actual_seed)
    loader = DataLoader(train, batch_size=batch_size, shuffle=True, num_workers=0,
                        generator=torch.Generator().manual_seed(actual_seed + 12345))
    optimizer = torch.optim.Adam(model.parameters(), lr=1e-3, weight_decay=0)
    history = []
    torch.cuda.reset_peak_memory_stats()
    for epoch in range(1, epochs + 1):
        lr = 1e-3 if epoch <= 3 else 1e-4 if epoch <= 9 else 1e-5
        for group in optimizer.param_groups:
            group["lr"] = lr
        model.train()
        total = correct = steps = 0
        loss_sum = 0.0
        started = time.perf_counter()
        for batch_index, (x, y) in enumerate(loader):
            if smoke and batch_index >= int(config.get("smoke_max_train_batches", 2)):
                break
            x, y = x.cuda(), y.cuda()
            optimizer.zero_grad(set_to_none=True)
            logits = model(x)
            loss = F.cross_entropy(logits, y)
            if not torch.isfinite(loss):
                raise ValueError("Nonfinite neural training loss")
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 5)
            optimizer.step()
            total += len(y)
            correct += int((logits.argmax(1) == y).sum())
            loss_sum += float(loss.detach()) * len(y)
            steps += 1
        history.append(dict(epoch=epoch, learning_rate=lr, train_loss=loss_sum / total,
                            train_accuracy=correct / total, optimizer_steps=steps,
                            examples_seen=total, seconds=time.perf_counter() - started))
        pd.DataFrame(history).to_csv(folder / "history.csv", index=False)
        print(f"GPU{os.environ.get('CUDA_VISIBLE_DEVICES')} S{subject:02} seed{seed} {fold_tag} "
              f"{arm} epoch{epoch}/{epochs} train={correct / total:.4f}", flush=True)
    temporary = checkpoint.with_name(checkpoint.name + ".tmp")
    torch.save(model.state_dict(), temporary)
    temporary.replace(checkpoint)
    json_write(manifest_path, dict(**fit_spec, signature=signature, success=True, state="fit_complete",
                                  status="complete", epochs_run=epochs,
                                  checkpoint_sha256=file_hash(checkpoint),
                                  parameter_count=int(model.count_params()),
                                  gpu=os.environ.get("CUDA_VISIBLE_DEVICES"), gpu_name=torch.cuda.get_device_name(0),
                                  peak_cuda_memory_bytes=int(torch.cuda.max_memory_allocated()),
                                  optimizer="Adam", weight_decay=0, gradient_clip=5,
                                  augmentation=False, checkpoint_selection="fixed_final_epoch",
                                  training_seconds=sum(row["seconds"] for row in history),
                                  smoke_label="NONRESEARCH SMOKE" if smoke else None))
    del optimizer, loader
    return model


def gpu_preflight(gpu, output):
    import torch
    from ablation_model import AblationC1
    if torch.cuda.device_count() != 1:
        raise RuntimeError("Worker requires exactly one visible CUDA GPU, pinned by launcher before import")
    if os.environ.get("CUDA_VISIBLE_DEVICES") != str(gpu):
        raise RuntimeError("CUDA_VISIBLE_DEVICES does not match assigned physical GPU")
    rows = []
    for arm in EXPERTS:
        model = AblationC1(arm, 0.65).cuda()
        x = torch.randn(2, 120, 400, device="cuda")
        if "S" in arm:
            model.fit_spectral_scaler([x])
        logits = model(x)
        loss = torch.nn.functional.cross_entropy(logits, torch.tensor([0, 16], device="cuda"))
        loss.backward()
        if logits.shape != (2, 17) or not torch.isfinite(logits).all():
            raise RuntimeError("Neural shape preflight failed")
        if not all(parameter.grad is not None and torch.isfinite(parameter.grad).all()
                   for parameter in model.parameters()):
            raise RuntimeError("Neural gradient preflight failed")
        rows.append(dict(arm=arm, shape=list(logits.shape), gradients_finite=True))
        del model, x, logits, loss
    torch.cuda.empty_cache()
    json_write(Path(output) / f"gpu{gpu}_preflight.json",
               dict(success=True, physical_gpu=gpu, visible_devices=os.environ["CUDA_VISIBLE_DEVICES"],
                    gpu_name=torch.cuda.get_device_name(0), pid=os.getpid(), checks=rows))


def metadata_indices(full_meta, subset_meta):
    full_index = pd.MultiIndex.from_frame(full_meta[KEYS])
    query = pd.MultiIndex.from_frame(subset_meta[KEYS])
    if not full_index.is_unique or not query.is_unique:
        raise ValueError("Duplicate window identity")
    selected = full_index.get_indexer(query)
    if (selected < 0).any():
        raise ValueError("Subset window identity absent from feature cache")
    return selected


def export_meta_results(folder, metadata, y, predictions, diagnostics, si_probs, wsi_probs):
    """Explicit window/trial/gesture/phase tables for all fusion methods and controls."""
    folder = Path(folder)
    folder.mkdir(parents=True, exist_ok=True)
    all_predictions = dict(predictions, control_si=si_probs, control_wsi=wsi_probs)
    metrics, cases = [], []
    baseline = si_probs.argmax(axis=1)
    for method, probabilities in all_predictions.items():
        probabilities = np.asarray(probabilities)
        row = dict(method=method, **probability_metrics(probabilities, y))
        row["probability_source"] = "uncalibrated_control_softmax" if method.startswith("control_") else "calibrated_expert_fusion"
        for column in ("subject", "seed"):
            if column in metadata and metadata[column].nunique() == 1:
                row[column] = int(metadata[column].iloc[0])
        predicted = probabilities.argmax(axis=1)
        row["recovered_vs_si"] = int(((baseline != y) & (predicted == y)).sum())
        row["harmed_vs_si"] = int(((baseline == y) & (predicted != y)).sum())
        metrics.append(row)
        cases.append(metadata.assign(method=method, prediction=predicted + 1,
                                     correct=predicted == y, confidence=probabilities.max(axis=1),
                                     recovered_vs_si=(baseline != y) & (predicted == y),
                                     harmed_vs_si=(baseline == y) & (predicted != y)))
        confusion = pd.crosstab(pd.Series(y + 1, name="gesture"),
                                pd.Series(predicted + 1, name="prediction")).reindex(
                                    index=range(1, 18), columns=range(1, 18), fill_value=0)
        confusion.to_csv(folder / f"confusion_{method}.csv")
    frame = pd.concat(cases, ignore_index=True)
    frame.to_csv(folder / "per_window_predictions.csv", index=False)
    pd.DataFrame(metrics).to_csv(folder / "method_metrics.csv", index=False)
    for name, keys in [("subject", ["subject"]), ("gesture", ["subject", "gesture"]),
                       ("repetition", ["subject", "gesture", "native_repetition"]),
                       ("phase", ["subject", "gesture", "native_repetition", "phase"])]:
        summary = frame.groupby(["method"] + keys, dropna=False).agg(
            windows=("correct", "size"), correct=("correct", "sum"),
            recovered_vs_si=("recovered_vs_si", "sum"), harmed_vs_si=("harmed_vs_si", "sum")).reset_index()
        summary["wrong"] = summary.windows - summary.correct
        summary["accuracy"] = summary.correct / summary.windows
        summary.to_csv(folder / f"per_{name}_errors.csv", index=False)
    atomic_npz(folder / "fusion_probabilities.npz", **{key: np.asarray(value, np.float32)
                                                      for key, value in all_predictions.items()}, y=y)
    flat_diagnostics = {}
    def flatten(prefix, value):
        if isinstance(value, dict):
            for key, child in value.items():
                flatten(f"{prefix}_{key}" if prefix else key, child)
        elif isinstance(value, (np.ndarray, list, tuple)):
            array = np.asarray(value)
            if array.dtype.kind in "biuf" and array.ndim and array.shape[0] == len(y):
                flat_diagnostics[prefix] = array
    flatten("", diagnostics)
    atomic_npz(folder / "reliability_diagnostics.npz", **flat_diagnostics)
    return metrics


def run_subject(subject, seed, config, output, run_signature):
    import torch
    from ablation_model import AblationC1
    import tc_support as support
    import brb_meta
    root = Path(output) / f"S{subject:02}" / f"seed_{seed}"
    root.mkdir(parents=True, exist_ok=True)
    raw_root = root / "source_inventory"
    raw_root.mkdir(exist_ok=True)
    source = Path(config["input_root"]) if config.get("input_root") else support.find_input()
    segments, inventory, _ignored_mean, _ignored_std = support.load_subject(source, subject, raw_root)
    json_write(raw_root / "SCALER_NOT_USED.json", dict(
        explanation="tc_support.load_subject convenience scaler is never used by BRB worker; fold scalers are recomputed",
        unused_scaler="input_scaler.npz"))
    smoke = bool(config.get("smoke", False))
    window_options = dict(smoke=smoke, smoke_cap=int(config.get("smoke_windows_per_trial", 32)))
    complete_inventory = fold_inventory(inventory, TRAIN_REPS, TEST_REPS)
    final_mean, final_std, final_provenance = fit_input_scaler(segments, complete_inventory, TRAIN_REPS)
    development = make_windows(segments, complete_inventory, final_mean, final_std, "train", **window_options)
    development.meta.to_csv(root / "development_windows.csv", index=False)
    development_raw = dataset_descriptors(development)
    atomic_npz(root / "development_physical_features.npz", features=development_raw)
    oof_logits = np.full((len(development), 3, 17), np.nan, dtype=np.float32)
    oof_shifts = np.full((len(development), 3), np.nan, dtype=np.float32)
    covered = np.zeros(len(development), dtype=int)
    fold_records = []
    cap = int(config.get("reference_cap_per_trial", 128))
    for held in TRAIN_REPS:
        train_reps = tuple(rep for rep in TRAIN_REPS if rep != held)
        fold = root / "oof" / f"held_rep_{held}"
        fold.mkdir(parents=True, exist_ok=True)
        subset = fold_inventory(inventory, train_reps, [held])
        mean, std, provenance = fit_input_scaler(segments, subset, train_reps)
        atomic_npz(fold / "input_scaler.npz", mean=mean, std=std)
        json_write(fold / "input_scaler_provenance.json", provenance)
        train = make_windows(segments, subset, mean, std, "train", **window_options)
        evaluation = make_windows(segments, subset, mean, std, "evaluation", **window_options)
        train.meta.to_csv(fold / "train_windows.csv", index=False)
        evaluation.meta.to_csv(fold / "held_windows.csv", index=False)
        train_indices = metadata_indices(development.meta, train.meta)
        held_indices = metadata_indices(development.meta, evaluation.meta)
        reference, selected = fit_shift_reference(development_raw[train_indices], train.meta, train_reps, cap)
        json_write(fold / "shift_reference.json", reference)
        train.meta.iloc[selected].to_csv(fold / "reference_windows.csv", index=False)
        oof_shifts[held_indices] = apply_shift_reference(development_raw[held_indices], reference)
        for arm_index, arm in enumerate(EXPERTS):
            model = train_fit(arm, seed, subject, train, fold / arm, f"oof_held_{held}", run_signature, config)
            logits = model_predict(model, evaluation, int(config.get("batch_size", 512)))
            save_predictions(fold / arm / "held", evaluation, logits)
            oof_logits[held_indices, arm_index] = logits
            del model
            gc.collect()
            torch.cuda.empty_cache()
        covered[held_indices] += 1
        fold_records.append(dict(held_repetition=held, training_repetitions=list(train_reps),
                                 training_window_sha256=window_hash(train.meta),
                                 held_window_sha256=window_hash(evaluation.meta),
                                 input_scaler_sha256=file_hash(fold / "input_scaler.npz"),
                                 shift_reference_sha256=reference["sha256"]))
        del train, evaluation
    if not np.all(covered == 1) or not np.isfinite(oof_logits).all() or not np.isfinite(oof_shifts).all():
        raise RuntimeError("OOF predictions do not cover every development window exactly once")
    oof_meta = development.meta.assign(oof_held_repetition=development.meta.native_repetition)
    oof_meta.to_csv(root / "oof_metadata.csv", index=False)
    atomic_npz(root / "oof_bundle.npz", logits=oof_logits, y=development.y,
               repetitions=development.meta.native_repetition.to_numpy(), shifts=oof_shifts,
               physical_features=development_raw)
    json_write(root / "oof_provenance.json", dict(arm_order=list(EXPERTS), folds=fold_records,
                                                test_repetitions_used=False,
                                                development_window_sha256=window_hash(development.meta)))
    # All calibration/gating/BRB fitting occurs here, before even building test windows.
    meta_folder = root / "meta"
    meta_folder.mkdir(exist_ok=True)
    meta_model = brb_meta.fit_meta(oof_logits, development.y,
                                   development.meta.native_repetition.to_numpy(), oof_shifts,
                                   meta_folder, metadata=dict(subject=subject, seed=seed,
                                   arm_order=list(EXPERTS), run_signature=run_signature,
                                   fold_provenance=fold_records, smoke=smoke,
                                   window_metadata_path="../oof_metadata.csv",
                                   window_metadata_sha256=file_hash(root / "oof_metadata.csv"),
                                   development_windows=len(oof_meta)))
    json_write(meta_folder / "locked_meta_model.json", meta_model)
    locked_meta_hash = file_hash(meta_folder / "locked_meta_model.json")
    json_write(meta_folder / "LOCKED_BEFORE_TEST.json", dict(
        model_sha256=locked_meta_hash, test_predictions_available=False,
        fit_inputs="training-only neural OOF logits/labels/repetitions and fold-fitted shifts",
        test_usage="secondary descriptive evaluation; hypotheses already informed by previous test results"))
    final_folder = root / "final"
    final_folder.mkdir(exist_ok=True)
    atomic_npz(final_folder / "input_scaler.npz", mean=final_mean, std=final_std)
    json_write(final_folder / "input_scaler_provenance.json", final_provenance)
    final_reference, selected = fit_shift_reference(development_raw, development.meta, TRAIN_REPS, cap)
    json_write(final_folder / "shift_reference.json", final_reference)
    development.meta.iloc[selected].to_csv(final_folder / "reference_windows.csv", index=False)
    for arm in FINAL_ARMS:
        model = train_fit(arm, seed, subject, development, final_folder / arm, "final", run_signature, config)
        del model
        gc.collect()
        torch.cuda.empty_cache()
    checkpoint_hashes = {arm: file_hash(final_folder / arm / "final_model.pt") for arm in FINAL_ARMS}
    json_write(root / "ALL_MODELS_LOCKED_BEFORE_TEST.json", dict(meta_model_sha256=locked_meta_hash,
                                                               checkpoint_sha256=checkpoint_hashes,
                                                               final_shift_reference_sha256=final_reference["sha256"]))
    test = make_windows(segments, complete_inventory, final_mean, final_std, "evaluation", **window_options)
    test.meta["seed"] = int(seed)
    if set(test.meta.native_repetition) != set(TEST_REPS):
        raise RuntimeError("Outer test split must contain only repetitions 2/5")
    test.meta.to_csv(root / "test_metadata.csv", index=False)
    test_raw = dataset_descriptors(test)
    test_shifts = apply_shift_reference(test_raw, final_reference)
    test_logits = np.empty((len(test), 3, 17), dtype=np.float32)
    controls = {}
    for arm in FINAL_ARMS:
        model = AblationC1(arm, config.get("dropout", 0.65)).cuda()
        model.load_state_dict(torch.load(final_folder / arm / "final_model.pt", map_location="cuda", weights_only=True))
        logits = model_predict(model, test, int(config.get("batch_size", 512)))
        probs = save_predictions(final_folder / arm / "test", test, logits)
        if arm in EXPERTS:
            test_logits[:, EXPERTS.index(arm)] = logits
        else:
            controls[arm] = probs
        del model
        gc.collect()
        torch.cuda.empty_cache()
    if file_hash(meta_folder / "locked_meta_model.json") != locked_meta_hash:
        raise RuntimeError("Meta model changed after locking")
    predictions = brb_meta.predict_meta(meta_model, test_logits, test_shifts)
    diagnostics = brb_meta.predict_diagnostics(meta_model, test_logits, test_shifts)
    results = export_meta_results(root / "results", test.meta, test.y, predictions, diagnostics,
                                  controls["SI"], controls["WSI"])
    json_write(root / "meta_completion.json", dict(subject=subject, seed=seed, success=True,
                                                  meta_model_sha256=locked_meta_hash,
                                                  methods=[row["method"] for row in results],
                                                  smoke=smoke))
    atomic_npz(root / "subject_bundle.npz", oof_logits=oof_logits, oof_y=development.y,
               oof_repetitions=development.meta.native_repetition.to_numpy(), oof_shifts=oof_shifts,
               test_logits=test_logits, test_y=test.y, test_shifts=test_shifts,
               test_physical_features=test_raw, control_si=controls["SI"], control_wsi=controls["WSI"])
    complete = dict(success=True, subject=subject, seed=seed, neural_fits=17,
                    oof_fits=12, final_fits=5, arm_order=list(EXPERTS),
                    run_signature=run_signature, meta_model_sha256=locked_meta_hash,
                    test_window_sha256=window_hash(test.meta), test_windows=len(test),
                    smoke=smoke, smoke_label="NONRESEARCH SMOKE" if smoke else None,
                    methods=[row["method"] for row in results])
    json_write(root / "SUBJECT_COMPLETE.json", complete)
    print(f"SUBJECT_COMPLETE S{subject:02} seed{seed} neural_fits=17", flush=True)
    del development, test, segments
    gc.collect()
    torch.cuda.empty_cache()
    return complete


def main(argv=None):
    parser = argparse.ArgumentParser(description=__doc__)
    parser.add_argument("--config", type=Path, required=True)
    parser.add_argument("--gpu-id", type=int, required=True)
    parser.add_argument("--subjects", required=True, help="comma-separated subject IDs assigned to this GPU")
    parser.add_argument("--output", type=Path, required=True)
    args = parser.parse_args(argv)
    args.output.mkdir(parents=True, exist_ok=True)
    try:
        config = json.loads(args.config.read_text(encoding="utf-8-sig"))
        validate_config(config)
        subjects = [int(value) for value in args.subjects.split(",")]
        if len(set(subjects)) != len(subjects) or not set(subjects).issubset(config["subjects"]):
            raise ValueError("Worker subjects must be a unique subset of configured subjects")
        # The launcher sets CUDA_VISIBLE_DEVICES before this process imports torch.
        import torch
        import tc_support as support
        torch.set_num_threads(int(config.get("cpu_threads_per_worker", 2)))
        support.Settings.TRAIN_REPS = list(TRAIN_REPS)
        support.Settings.TEST_REPS = list(TEST_REPS)
        support.Settings.FS, support.Settings.WINDOW, support.Settings.STEP = 2000, 400, 20
        support.Settings.BATCH_SIZE, support.Settings.DROPOUT = 512, 0.65
        scripts = Path(__file__).resolve().parent
        code_hashes = {path.name: file_hash(path) for path in sorted(scripts.glob("*.py"))}
        signature = digest(dict(config=config, code=code_hashes))
        run_record = args.output / f"worker_{args.gpu_id}_configuration.json"
        if run_record.exists():
            existing = json.loads(run_record.read_text(encoding="utf-8"))
            if existing["run_signature"] != signature:
                raise RuntimeError("Output directory belongs to a different code/configuration")
        json_write(run_record, dict(config=config, code_sha256=code_hashes, run_signature=signature,
                                    assigned_subjects=subjects, physical_gpu=args.gpu_id))
        gpu_preflight(args.gpu_id, args.output)
        completed = []
        for subject in subjects:
            for seed in config["seeds"]:
                completed.append(run_subject(subject, int(seed), config, args.output, signature))
        json_write(args.output / f"worker_{args.gpu_id}_complete.json",
                   dict(success=True, physical_gpu=args.gpu_id, subjects=subjects,
                        neural_fits=sum(record["neural_fits"] for record in completed),
                        run_signature=signature, completions=completed))
    except Exception:
        (args.output / f"gpu{args.gpu_id}_failure.txt").write_text(traceback.format_exc(), encoding="utf-8")
        raise


if __name__ == "__main__":
    main()


## brb_meta.py

Frozen temperature/global-weight fitting and generic reliability indicators. Its historical fit_meta routine is not invoked.

In [ ]:
%%writefile /kaggle/working/db7_suite_source/brb_meta.py
"""Training-only reliability fusion for DB7 W/S/I expert logits.

Labels are zero based. fit_meta accepts OOF predictions for repetitions 1/3/4/6
only. Repetition 6 is reserved for reliability calibration, not model fitting,
temperature selection or global fusion weights. The caller must supply shifts
computed against each OOF base model's own training-only signal references.

This is nested development, NOT independent meta cross-validation: OOF base
models may share training recordings. Outer test predictions never enter fit.
"""
from __future__ import annotations

import csv
import hashlib
import json
from pathlib import Path

import numpy as np
from scipy.optimize import minimize
from scipy.special import expit, logit, logsumexp, softmax

VERSION = "db7-brb-meta-v1"
EXPERTS = ("W", "S", "I")
FIT_REPS = (1, 3, 4)
CAL_REP = 6
EPS = 1e-9
HEAD_L2 = 0.01
CAL_L2 = 0.01
ALPHA_L2 = 0.005
MIN_EVENTS = 5
MAXITER = 180
RULE_BITS = np.array([[int(x) for x in f"{r:03b}"] for r in range(8)])


def _inputs(logits, shifts, y=None, repetitions=None):
    z = np.asarray(logits, dtype=np.float64)
    s = np.asarray(shifts, dtype=np.float64)
    if z.ndim != 3 or z.shape[1:] != (3, 17) or len(z) == 0:
        raise ValueError("logits must have nonempty shape [N,3,17]")
    if s.shape != z.shape[:2] or not np.isfinite(z).all() or not np.isfinite(s).all():
        raise ValueError("finite shifts [N,3] and logits required")
    if np.any((s < 0) | (s > 1)):
        raise ValueError("training-reference shifts must be in [0,1]")
    if y is None:
        return z, s
    yy = np.asarray(y)
    rr = np.asarray(repetitions)
    if yy.shape != (len(z),) or rr.shape != yy.shape:
        raise ValueError("y and repetitions must have shape [N]")
    if not np.isin(yy, np.arange(17)).all():
        raise ValueError("y must contain zero-based labels 0..16")
    if not np.isin(rr, (*FIT_REPS, CAL_REP)).all():
        raise ValueError("fit_meta accepts only training repetitions 1,3,4,6; test forbidden")
    if set(rr.tolist()) != {1, 3, 4, 6}:
        raise ValueError("all meta-fit repetitions 1/3/4 and calibration repetition 6 required")
    return z, s, yy.astype(np.int64), rr.astype(np.int64)


def _group_weights(groups):
    """Equal repetition weight; do not treat differing window counts as trials."""
    groups = np.asarray(groups)
    levels, counts = np.unique(groups, return_counts=True)
    return np.array([1.0 / (len(levels) * counts[np.searchsorted(levels, g)]) for g in groups])


def _binary_loss(target, predicted, weights):
    p = np.clip(predicted, EPS, 1 - EPS)
    return float(-np.sum(weights * (target * np.log(p) + (1 - target) * np.log1p(-p))))


def _optimization(result):
    return {"success": bool(result.success), "message": str(result.message),
            "iterations": int(result.nit), "objective": float(result.fun)}


def fit_temperatures(logits, y, groups):
    """Positive scalar temperature per expert, using supplied development rows."""
    weights = _group_weights(groups)
    temperatures, diagnostics = [], []
    for j in range(3):
        z = logits[:, j]
        def objective(theta):
            zz = z / np.exp(theta[0])
            p = softmax(zz, axis=1)
            loss = np.sum(weights * (logsumexp(zz, axis=1) - zz[np.arange(len(y)), y]))
            grad = np.sum(weights * (zz[np.arange(len(y)), y] - np.sum(p * zz, axis=1)))
            return float(loss + 0.001 * theta[0] ** 2), np.array([grad + .002 * theta[0]])
        opt = minimize(objective, [0.], jac=True, method="L-BFGS-B",
                       bounds=[(np.log(.05), np.log(20.))], options={"maxiter": MAXITER})
        if not np.isfinite(opt.fun):
            raise RuntimeError("nonfinite temperature optimization")
        temperatures.append(float(np.exp(opt.x[0])))
        diagnostics.append(_optimization(opt))
    return temperatures, diagnostics


def _probabilities(logits, temperatures):
    return softmax(logits / np.asarray(temperatures)[None, :, None], axis=2)


def indicators(probabilities, shifts):
    """Expert entropy, mean pairwise TV and precomputed training deviation."""
    p = np.asarray(probabilities, dtype=float)
    entropy = -np.sum(p * np.log(np.clip(p, EPS, 1)), axis=2) / np.log(17.)
    disagreement = np.zeros(p.shape[:2])
    for j in range(3):
        disagreement[:, j] = sum(.5 * np.abs(p[:, j] - p[:, k]).sum(1)
                                for k in range(3) if k != j) / 2
    return np.clip(np.stack([entropy, disagreement, shifts], axis=-1), 0, 1)


def rule_activations(q):
    """Product reference matching: [...,3] -> [...,8], sum exactly one."""
    q = np.asarray(q, dtype=float)
    if q.shape[-1] != 3 or not np.isfinite(q).all() or np.any((q < 0) | (q > 1)):
        raise ValueError("rule indicators must be finite [...,3] in [0,1]")
    a = np.prod(np.where(RULE_BITS, q[..., None, :], 1 - q[..., None, :]), axis=-1)
    return a / a.sum(axis=-1, keepdims=True)


def rimer_correct(activations, correct_beliefs, return_jacobian=False):
    """Analytical ER/RIMER for complete binary rule conclusions.

    A_n=prod(1-w+w*beta_n), B=prod(1-w), beta_n=(A_n-B)/(A0+A1-2B).
    Normalized activation is rule evidence weight. Returned jacobian is with
    respect to each rule's correctness belief (NOT its logit).
    """
    w = np.asarray(activations, dtype=float)
    b = np.asarray(correct_beliefs, dtype=float)
    if w.shape[-1] != 8 or b.shape != (8,):
        raise ValueError("eight activations and eight correctness beliefs required")
    if not np.isfinite(w).all() or not np.isfinite(b).all() or np.any((b < 0) | (b > 1)):
        raise ValueError("invalid ER beliefs")
    if np.any((w < 0) | (w > 1)) or not np.allclose(w.sum(-1), 1):
        raise ValueError("ER activations must be normalized")
    f1 = 1 - w + w * b
    f0 = 1 - w + w * (1 - b)
    a1, a0, bb = f1.prod(-1), f0.prod(-1), (1 - w).prod(-1)
    denom = a1 + a0 - 2 * bb
    if np.any(denom <= 0):
        raise FloatingPointError("degenerate ER normalization")
    p = np.clip((a1 - bb) / denom, 0, 1)
    if not return_jacobian:
        return p
    # Product excluding each factor handles exact zero factors at rule vertices.
    da1 = np.stack([w[..., k] * np.delete(f1, k, axis=-1).prod(-1) for k in range(8)], -1)
    da0 = -np.stack([w[..., k] * np.delete(f0, k, axis=-1).prod(-1) for k in range(8)], -1)
    jac = (da1 * denom[..., None] - (a1 - bb)[..., None] * (da1 + da0)) / denom[..., None] ** 2
    return p, jac


def _fit_alpha(p, y, weights):
    true_p = p[np.arange(len(p))[:, None], np.arange(3)[None, :], y[:, None]]
    def objective(theta):
        alpha = softmax(theta)
        mixture = np.clip(true_p @ alpha, EPS, 1)
        loss = -np.sum(weights * np.log(mixture)) + ALPHA_L2 * np.sum(theta ** 2)
        da = -np.sum(weights[:, None] * true_p / mixture[:, None], axis=0)
        grad = alpha * (da - alpha @ da) + 2 * ALPHA_L2 * theta
        return float(loss), grad
    opt = minimize(objective, np.zeros(3), jac=True, method="L-BFGS-B",
                   bounds=[(-6, 6)] * 3, options={"maxiter": MAXITER})
    return softmax(opt.x).tolist(), _optimization(opt)


def _raw_head(head, q):
    if head["kind"] == "constant":
        return np.full(len(q), head["value"])
    if head["kind"] == "logistic":
        return expit(np.column_stack([np.ones(len(q)), q - .5]) @ np.asarray(head["parameters"]))
    qq = q.copy()
    if head["kind"] == "brb_no_shift":
        qq[:, 2] = .5
    a = rule_activations(qq)
    beliefs = expit(head["parameters"])
    return a @ beliefs if head["kind"] == "sugeno" else rimer_correct(a, beliefs)


def _fit_head(kind, q, target, weights):
    prior = float((np.sum(target) + .5) / (len(target) + 1))
    if min(int(target.sum()), int((1 - target).sum())) < MIN_EVENTS:
        return {"kind": "constant", "requested_kind": kind, "value": prior,
                "fallback": "fewer than five correct or incorrect examples"}
    if kind == "logistic":
        x = np.column_stack([np.ones(len(q)), q - .5])
        center = np.array([logit(prior), 0, 0, 0])
        def objective(theta):
            raw = expit(x @ theta)
            loss = _binary_loss(target, raw, weights) + HEAD_L2 * np.mean((theta - center) ** 2)
            grad = x.T @ (weights * (raw - target)) + 2 * HEAD_L2 * (theta - center) / len(theta)
            return loss, grad
    else:
        qq = q.copy()
        if kind == "brb_no_shift":
            qq[:, 2] = .5
        a = rule_activations(qq)
        center = np.full(8, logit(prior))
        def objective(theta):
            beliefs = expit(theta)
            if kind == "sugeno":
                raw, jac = a @ beliefs, a
            else:
                raw, jac = rimer_correct(a, beliefs, return_jacobian=True)
            pp = np.clip(raw, EPS, 1 - EPS)
            derivative = weights * (pp - target) / (pp * (1 - pp))
            grad = (derivative @ jac) * beliefs * (1 - beliefs)
            loss = _binary_loss(target, pp, weights) + HEAD_L2 * np.mean((theta - center) ** 2)
            grad += 2 * HEAD_L2 * (theta - center) / len(theta)
            return loss, grad
    opt = minimize(objective, center, jac=True, method="L-BFGS-B", bounds=[(-10, 10)] * len(center),
                   options={"maxiter": MAXITER, "ftol": 1e-9})
    if not np.isfinite(opt.fun) or not np.isfinite(opt.x).all():
        raise RuntimeError(f"nonfinite {kind} fit")
    return {"kind": kind, "parameters": opt.x.tolist(), "optimization": _optimization(opt), "prior": prior}


def _fit_calibration(raw, target):
    """Monotone logit-affine reliability calibration on repetition 6 only."""
    if len(target) < 20:
        return {"slope": 1., "intercept": 0., "fallback": "fewer than 20 calibration rows"}
    x = logit(np.clip(raw, 1e-5, 1 - 1e-5))
    sparse = min(int(target.sum()), int((1 - target).sum())) < MIN_EVENTS
    def objective(theta):
        slope, intercept = theta
        p = expit(slope * x + intercept)
        weights = np.full(len(target), 1 / len(target))
        loss = _binary_loss(target, p, weights) + CAL_L2 * ((slope - 1) ** 2 + intercept ** 2)
        residual = p - target
        grad = np.array([np.mean(residual * x) + 2 * CAL_L2 * (slope - 1),
                         np.mean(residual) + 2 * CAL_L2 * intercept])
        return loss, grad
    opt = minimize(objective, [1., 0.], jac=True, method="L-BFGS-B",
                   bounds=[(1., 1.) if sparse else (0., 5.), (-8., 8.)], options={"maxiter": MAXITER})
    return {"slope": float(opt.x[0]), "intercept": float(opt.x[1]), "optimization": _optimization(opt),
            "fallback": "intercept-only: fewer than five events in one outcome" if sparse else None}


def _calibrated(raw, calibration):
    return expit(calibration["slope"] * logit(np.clip(raw, 1e-5, 1 - 1e-5)) + calibration["intercept"])


def _weights(alpha, reliability):
    unnormalized = np.asarray(alpha)[None, :] * np.clip(reliability, 0, 1)
    denominator = unnormalized.sum(1, keepdims=True)
    return np.divide(unnormalized, denominator, out=np.broadcast_to(alpha, unnormalized.shape).copy(), where=denominator > EPS)


def predict_diagnostics(model, logits, shifts):
    z, s = _inputs(logits, shifts)
    if model.get("version") != VERSION:
        raise ValueError("unsupported meta model version")
    p = _probabilities(z, model["temperatures"])
    q = indicators(p, s)
    reliabilities = {"confidence_weight": p.max(2)}
    raw_reliabilities = {}
    for name, heads in model["heads"].items():
        raw = np.column_stack([_raw_head(heads[j], q[:, j]) for j in range(3)])
        raw_reliabilities[name] = raw
        reliabilities[name] = np.column_stack([_calibrated(raw[:, j], model["reliability_calibrations"][name][j]) for j in range(3)])
    weights = {name: _weights(model["alpha"], r) for name, r in reliabilities.items()}
    return {"expert_probabilities": p, "indicators": q, "raw_reliabilities": raw_reliabilities,
            "reliabilities": reliabilities, "weights": weights, "rule_activations": rule_activations(q)}


def predict_meta(model, logits, shifts):
    d = predict_diagnostics(model, logits, shifts)
    p = d["expert_probabilities"]
    result = {f"expert_{name.lower()}": p[:, j] for j, name in enumerate(EXPERTS)}
    result["mean"] = p.mean(1)
    result["global_weight"] = np.sum(p * np.asarray(model["alpha"])[None, :, None], axis=1)
    result.update({name: np.sum(p * w[:, :, None], axis=1) for name, w in d["weights"].items()})
    return result


def _metrics(y, p):
    confidence, predicted = p.max(1), p.argmax(1)
    correct = predicted == y
    onehot = np.eye(p.shape[1])[y]
    ece = 0.
    for low in np.arange(0, 1, .1):
        mask = (confidence >= low) & (confidence < low + .1 if low < .9 else confidence <= 1)
        if mask.any():
            ece += mask.mean() * abs(correct[mask].mean() - confidence[mask].mean())
    return {"rows": int(len(y)), "accuracy": float(correct.mean()),
            "nll": float(-np.log(np.clip(p[np.arange(len(y)), y], EPS, 1)).mean()),
            "brier": float(np.sum((p - onehot) ** 2, axis=1).mean()), "ece10": float(ece)}


def _write_csv(path, rows):
    if not rows:
        return
    with path.open("w", newline="", encoding="utf-8") as f:
        writer = csv.DictWriter(f, fieldnames=list(rows[0]))
        writer.writeheader()
        writer.writerows(rows)


def fit_meta(logits, y, repetitions, shifts, output: Path, metadata=None):
    """Fit and save a JSON-serializable meta model; never supply test rows.

    1/3/4: final temperatures, alpha, reliability heads. Head training inputs use
    leave-one-repetition crossfitted temperatures. 6: monotone scalar reliability
    calibration only. Final temperature is frozen before looking at rep 6.
    """
    z, s, yy, rr = _inputs(logits, shifts, y, repetitions)
    fit = np.isin(rr, FIT_REPS)
    cal = rr == CAL_REP
    temperatures, tdiag = fit_temperatures(z[fit], yy[fit], rr[fit])
    p_final = _probabilities(z, temperatures)
    p_crossfit = np.zeros_like(z[fit])
    crossfits = []
    for held in FIT_REPS:
        inner_train = fit & (rr != held)
        temp, opt = fit_temperatures(z[inner_train], yy[inner_train], rr[inner_train])
        p_crossfit[rr[fit] == held] = _probabilities(z[fit & (rr == held)], temp)
        crossfits.append({"held_repetition": held, "fit_repetitions": sorted(set(rr[inner_train].tolist())),
                          "temperatures": temp, "optimization": opt})
    q_fit = indicators(p_crossfit, s[fit])
    q_cal = indicators(p_final[cal], s[cal])
    # Positive temperature cannot change an expert's class argmax.
    correctness_fit = (z[fit].argmax(2) == yy[fit, None]).astype(float)
    correctness_cal = (z[cal].argmax(2) == yy[cal, None]).astype(float)
    weights_fit = _group_weights(rr[fit])
    alpha, adiag = _fit_alpha(p_crossfit, yy[fit], weights_fit)
    model = {"version": VERSION, "expert_order": list(EXPERTS), "num_classes": 17,
             "temperatures": temperatures, "alpha": alpha, "heads": {}, "reliability_calibrations": {},
             "metadata": metadata or {}, "provenance": {
                 "meta_fit_repetitions": list(FIT_REPS), "reliability_calibration_repetition": CAL_REP,
                 "outer_test_repetitions_forbidden_at_fit": [2, 5], "fit_rows": int(fit.sum()), "calibration_rows": int(cal.sum()),
                 "counts_per_repetition": {str(r): int((rr == r).sum()) for r in sorted(set(rr.tolist()))},
                 "temperatures_crossfit": crossfits, "final_temperature_optimization": tdiag, "alpha_optimization": adiag,
                 "alpha_fit_input": "crossfitted-temperature probabilities from repetitions 1/3/4",
                 "fit_digest": hashlib.sha256(z[fit].tobytes() + yy[fit].tobytes() + s[fit].tobytes() + rr[fit].tobytes()).hexdigest(),
                 "calibration_digest": hashlib.sha256(z[cal].tobytes() + yy[cal].tobytes() + s[cal].tobytes()).hexdigest(),
                 "fixed_hyperparameters": {"head_l2": HEAD_L2, "calibration_l2": CAL_L2, "alpha_l2": ALPHA_L2, "min_events": MIN_EVENTS},
                 "rule_inputs": ["normalized_entropy", "mean_pairwise_total_variation", "training_reference_deviation"],
                 "caveat": "internal development; shared base-model training histories mean this is not independent meta CV"}}
    support_rows, rule_rows, reliability_rows, initial_rows = [], [], [], []
    for name in ("logistic", "sugeno", "brb", "brb_no_shift"):
        model["heads"][name], model["reliability_calibrations"][name] = [], []
        for j, expert in enumerate(EXPERTS):
            head = _fit_head(name, q_fit[:, j], correctness_fit[:, j], weights_fit)
            raw_cal = _raw_head(head, q_cal[:, j])
            calibration = _fit_calibration(raw_cal, correctness_cal[:, j])
            model["heads"][name].append(head)
            model["reliability_calibrations"][name].append(calibration)
            for stage, values in [("before", raw_cal), ("after", _calibrated(raw_cal, calibration))]:
                reliability_rows.append({"method": name, "expert": expert, "stage": stage, "repetition": 6,
                    "scope": "calibration fitting rows; not independent evaluation", "rows": len(raw_cal),
                    "observed_correctness": float(correctness_cal[:, j].mean()), "mean_reliability": float(values.mean()),
                    "brier_binary": float(np.mean((values - correctness_cal[:, j]) ** 2)),
                    "nll_binary": _binary_loss(correctness_cal[:, j], values, np.full(len(values), 1 / len(values)))})
            if name != "logistic":
                qq = q_fit[:, j].copy()
                if name == "brb_no_shift":
                    qq[:, 2] = .5
                a = rule_activations(qq)
                beliefs = np.full(8, head["value"]) if head["kind"] == "constant" else expit(head["parameters"])
                for rule, bits in enumerate(RULE_BITS):
                    initial = float(head.get("prior", head.get("value")))
                    initial_rows.append({"method": name, "expert": expert, "rule": rule,
                        "uncertainty": int(bits[0]), "disagreement": int(bits[1]), "deviation": int(bits[2]),
                        "belief_incorrect": 1-initial, "belief_correct": initial,
                        "initial_logit": float(logit(initial)), "rule_weight": 1.0,
                        "antecedent_weights": "1,1,1", "reference_values": "0,1"})
                    rule_rows.append({"method": name, "expert": expert, "rule": rule,
                        "uncertainty": int(bits[0]), "disagreement": int(bits[1]), "deviation": int(bits[2]),
                        "belief_incorrect": float(1 - beliefs[rule]), "belief_correct": float(beliefs[rule])})
                    for rep in FIT_REPS:
                        mask = rr[fit] == rep
                        mass = a[mask, rule]
                        support_rows.append({"method": name, "expert": expert, "rule": rule, "repetition": rep,
                            "window_count": int(mask.sum()), "activation_mass": float(mass.sum()),
                            "mean_activation": float(mass.mean()), "windows_activation_above_0_1": int((mass > .1).sum()),
                            "effective_windows_kish_correlated_not_trials": float(mass.sum() ** 2 / max(np.sum(mass ** 2), EPS)),
                            "weighted_correctness": float(mass @ correctness_fit[mask, j] / max(mass.sum(), EPS))})
    output = Path(output)
    output.mkdir(parents=True, exist_ok=True)
    (output / "meta_model.json").write_text(json.dumps(model, indent=2, allow_nan=False), encoding="utf-8")
    (output / "meta_provenance.json").write_text(json.dumps(model["provenance"], indent=2, allow_nan=False), encoding="utf-8")
    _write_csv(output / "meta_rule_initial.csv", initial_rows)
    _write_csv(output / "meta_rule_conclusions.csv", rule_rows)
    _write_csv(output / "meta_rule_support_per_repetition.csv", support_rows)
    _write_csv(output / "meta_reliability_calibration.csv", reliability_rows)
    pred = predict_meta(model, z, s)
    metrics = [{"method": name, "repetition": int(rep),
                "scope": "meta-head development fit" if rep in FIT_REPS else "reliability calibration fit",
                **_metrics(yy[rr == rep], pp[rr == rep])}
               for name, pp in pred.items() for rep in sorted(set(rr.tolist()))]
    _write_csv(output / "meta_development_metrics.csv", metrics)
    return model


## rule_core.py

Generalized rule reference matching, complete-belief ER, analytic optimization gradients, Sugeno/logistic controls and initial/final rule export.

In [ ]:
%%writefile /kaggle/working/db7_suite_source/rule_core.py
"""Complete-belief BRB/ER, Sugeno and logistic controls; analytic gradients.

All fitting inputs must be development data. Reference positions are fitted
inside each training fold. Consequent order is supplied explicitly by caller.
"""
import itertools
import numpy as np
from scipy.optimize import minimize
from scipy.special import softmax

EPS=1e-12

def references(q,counts):
    refs=[]
    for j,n in enumerate(counts):
        if n==2: refs.append([0.,1.])
        elif n==3: refs.append([0.,float(np.clip(np.median(q[:,j]),.1,.9)),1.])
        else: raise ValueError('Only prespecified two/three references supported')
    return refs

def matching(q,refs):
    q=np.asarray(q,float)
    assert q.ndim==2 and q.shape[1]==len(refs) and np.isfinite(q).all()
    assert np.min(q)>=0 and np.max(q)<=1
    bits=np.array(list(itertools.product(*[range(len(r)) for r in refs])))
    matches=[]
    for j,r in enumerate(refs):
        r=np.asarray(r);assert np.all(np.diff(r)>0)
        m=np.zeros((len(q),len(r)))
        lo=np.clip(np.searchsorted(r,q[:,j],side='right')-1,0,len(r)-2)
        fraction=(q[:,j]-r[lo])/(r[lo+1]-r[lo])
        m[np.arange(len(q)),lo]=1-fraction;m[np.arange(len(q)),lo+1]=fraction
        matches.append(m[:,bits[:,j]])
    return np.stack(matches,-1),bits

def aggregate(a,beta,kind='brb',gradient=None):
    """Return probabilities, optionally vector-Jacobian products dL/da,dL/dbeta."""
    if kind=='sugeno':
        p=a@beta
        return p if gradient is None else (gradient@beta.T,a.T@gradient)
    f=1-a[:,:,None]+a[:,:,None]*beta[None,:,:]
    products=np.prod(f,axis=1);ignorance=np.prod(1-a,axis=1)
    v=products-ignorance[:,None];den=v.sum(1)
    assert np.all(den>0)
    p=v/den[:,None]
    if gradient is None:return p
    dv=(gradient-(gradient*p).sum(1)[:,None])/den[:,None]
    # Softmax beliefs keep factors positive, even at exact reference vertices.
    df=dv[:,None,:]*products[:,None,:]/np.maximum(f,EPS)
    dbeta=(df*a[:,:,None]).sum(0)
    # Stable product excluding each rule handles exact a=1.
    prefix=np.cumprod(np.column_stack([np.ones(len(a)),1-a]),axis=1)
    suffix=np.cumprod(np.column_stack([1-a,np.ones(len(a))])[:,::-1],axis=1)[:,::-1]
    excluded=prefix[:,:-1]*suffix[:,1:]
    da=(df*(beta[None,:,:]-1)).sum(2)+dv.sum(1)[:,None]*excluded
    return da,dbeta

def fit(q,y,weights,counts,kind='brb',learn_weights=False,k=3,penalty=.01):
    q=np.asarray(q,float);y=np.asarray(y,int);weights=np.asarray(weights,float);weights/=weights.sum()
    assert len(q)==len(y) and np.isin(y,np.arange(k)).all()
    refs=references(q,counts);m,bits=matching(q,refs);nr=len(bits);d=q.shape[1]
    prior=np.bincount(y,weights=weights,minlength=k)+.01;prior/=prior.sum()
    if kind=='logistic':
        x=np.column_stack([np.ones(len(q)),q-.5]);center=np.zeros((d+1,k));center[0]=np.log(prior)
        def objective(theta):
            t=theta.reshape(d+1,k);p=softmax(x@t,axis=1)
            loss=-np.sum(weights*np.log(np.maximum(p[np.arange(len(y)),y],EPS)))
            g=p.copy();g[np.arange(len(y)),y]-=1;g*=weights[:,None]
            delta=t-center
            return loss+penalty*np.mean(delta**2),(x.T@g+2*penalty*delta/delta.size).ravel()
        initial=center.ravel();bounds=[(-12,12)]*len(initial)
    else:
        center=np.tile(np.log(prior),(nr,1));size=nr*k
        initial=np.r_[center.ravel(),np.zeros(nr+d)] if learn_weights else center.ravel()
        logm=np.log(np.maximum(m,EPS));valid=np.all(m>0,axis=2)
        def objective(theta):
            beta=softmax(theta[:size].reshape(nr,k),axis=1)
            rw=theta[size:size+nr] if learn_weights else np.zeros(nr)
            aw=np.exp(theta[size+nr:]) if learn_weights else np.ones(d)
            score=(logm*aw).sum(2)+rw;score=np.where(valid,score,-1e30);a=softmax(score,axis=1)
            p=aggregate(a,beta,kind);chosen=np.maximum(p[np.arange(len(y)),y],EPS)
            loss=-np.sum(weights*np.log(chosen));g=np.zeros_like(p);g[np.arange(len(y)),y]=-weights/chosen
            da,db=aggregate(a,beta,kind,g);gb=beta*(db-(db*beta).sum(1)[:,None])
            delta=theta[:size]-center.ravel();loss+=penalty*np.mean(delta**2)
            grad=gb.ravel()+2*penalty*delta/size
            if learn_weights:
                ds=a*(da-(da*a).sum(1)[:,None]);extra=theta[size:]
                gw=ds.sum(0);ga=np.einsum('nr,nrd->d',ds,logm)*aw
                loss+=penalty*np.mean(extra**2)
                grad=np.r_[grad,np.r_[gw,ga]+2*penalty*extra/len(extra)]
            return float(loss),grad
        bounds=[(-12,12)]*size+([(-2,2)]*nr+[(-1,1)]*d if learn_weights else [])
    result=minimize(objective,initial,jac=True,method='L-BFGS-B',bounds=bounds,options={'maxiter':180,'ftol':1e-9})
    if not np.isfinite(result.fun):raise RuntimeError('Non-finite rule optimization')
    return dict(kind=kind,learn_weights=learn_weights,refs=refs,bits=bits.tolist(),k=k,
                parameters=result.x.tolist(),initial=initial.tolist(),penalty=penalty,
                optimizer=dict(success=bool(result.success),message=str(result.message),iterations=int(result.nit),loss=float(result.fun)))

def predict(model,q,initial=False):
    t=np.asarray(model['initial' if initial else 'parameters']);k=model['k']
    if model['kind']=='logistic':return softmax(np.column_stack([np.ones(len(q)),q-.5])@t.reshape(-1,k),axis=1)
    m,bits=matching(q,model['refs']);nr=len(bits);beta=softmax(t[:nr*k].reshape(nr,k),axis=1)
    rw=t[nr*k:nr*k+nr] if model['learn_weights'] else np.zeros(nr)
    aw=np.exp(t[nr*k+nr:]) if model['learn_weights'] else np.ones(q.shape[1])
    scores=(np.log(np.maximum(m,EPS))*aw).sum(2)+rw
    a=softmax(np.where(np.all(m>0,axis=2),scores,-1e30),axis=1)
    return aggregate(a,beta,model['kind'])

def rule_rows(model,antecedents,consequents):
    if model['kind']=='logistic':return []
    nr=len(model['bits']);k=model['k'];rows=[]
    for state in ['initial','parameters']:
        t=np.asarray(model[state]);beta=softmax(t[:nr*k].reshape(nr,k),axis=1)
        for r,bits in enumerate(model['bits']):
            row=dict(state='initial' if state=='initial' else 'trained',rule=r+1)
            row.update({name:model['refs'][j][bits[j]] for j,name in enumerate(antecedents)})
            row.update({f'belief_{name}':float(beta[r,j]) for j,name in enumerate(consequents)})
            row['rule_weight']=float(np.exp(t[nr*k+r])) if model['learn_weights'] else 1.
            row.update({f'attribute_weight_{name}':float(np.exp(t[nr*k+nr+j])) if model['learn_weights'] else 1. for j,name in enumerate(antecedents)})
            rows.append(row)
    return rows


## suite_neural.py

Two isolated GPU workers, checkpoint/scaler identity checks, train-only physical prototypes and matched calibration/test caches.

In [ ]:
%%writefile /kaggle/working/db7_suite_source/suite_neural.py
"""Matched checkpoint prediction cache, two isolated GPU workers.

Seed42 reuses DB7-018 held-repetition-6 checkpoints. Seeds43/44 are new
13-epoch fits on1/3/4. Calibration6 and test2/5 always use the SAME model.
No calibration/test label selects a neural checkpoint or a preprocessing fit.
"""
from pathlib import Path
import argparse, gc, json, os, subprocess, sys, zipfile
import numpy as np
import pandas as pd
import brb_worker as w

SOURCE_SHA='65cf7bf7b914e0eb2f4178e96cffc74d27b6d42dacd694a6eaac42c4aec54eb0'

def source_archive():
    found=list(Path('/kaggle/input').rglob('db7_brb_full_20260915T013327619764Z.zip'))
    assert len(found)==1, 'Exactly one verified DB7-018 archive required'
    assert w.file_hash(found[0])==SOURCE_SHA, 'Source archive mismatch'
    return found[0]

def extract_checkpoints(archive,out):
    """Only authoritative held6 checkpoints/scalers, no ambiguous fallback."""
    out=Path(out);out.mkdir(parents=True,exist_ok=True)
    with zipfile.ZipFile(archive) as z:
        done=json.loads(z.read('completion.json'))
        assert done['success'] and done['neural_fits']==374
        for name in z.namelist():
            if '/oof/held_rep_6/' in name and name.endswith(('.pt','.json','.npz')):
                target=out/name
                assert target.resolve().is_relative_to(out.resolve())
                target.parent.mkdir(parents=True,exist_ok=True);target.write_bytes(z.read(name))
    for s in range(1,23):
        folds=list(out.glob(f'full/*_subjects/S{s:02d}/seed_42/oof/held_rep_6'))
        assert len(folds)==1, f'Missing or ambiguous S{s} held6 source'
        for arm in w.EXPERTS:
            folder=folds[0]/arm;m=json.loads((folder/'fit_manifest.json').read_text())
            assert m['success'] and m['epochs_run']==13
            assert sorted(m['training_repetitions'])==[1,3,4]
            assert m['held_out_repetition']==6
            assert w.file_hash(folder/'final_model.pt')==m['checkpoint_sha256']
    return out

def distances(raw,reference,centers):
    x=raw.astype(np.float64).copy()
    x[:,:12]=np.log(np.maximum(x[:,:12],reference['rms_floor']))
    x=(x-np.asarray(reference['center']))/np.asarray(reference['scale'])
    out=np.empty((len(x),3,17),np.float32)
    for j,(lo,hi) in enumerate([(0,12),(12,24),(24,132)]):
        for k in range(17):out[:,j,k]=np.sqrt(np.mean((x[:,lo:hi]-centers[k,lo:hi])**2,axis=1))
    return out

def subject_cache(subject,seed,source,out):
    import torch
    import tc_support as support
    from ablation_model import AblationC1
    root=Path(out)/f'seed_{seed}'/f'S{subject:02d}';root.mkdir(parents=True,exist_ok=True)
    inventory_dir=root/'inventory'
    inventory_dir.mkdir(parents=True,exist_ok=True)
    segments,inventory,_,_=support.load_subject(support.find_input(),subject,inventory_dir)
    inv=w.fold_inventory(inventory,[1,3,4],[6]);mean,std,prov=w.fit_input_scaler(segments,inv,[1,3,4])
    train=w.make_windows(segments,inv,mean,std,'train')
    raw=w.dataset_descriptors(train)
    reference,keep=w.fit_shift_reference(raw,train.meta,[1,3,4],128)
    transformed=raw.astype(np.float64).copy()
    transformed[:,:12]=np.log(np.maximum(transformed[:,:12],reference['rms_floor']))
    transformed=(transformed-np.asarray(reference['center']))/np.asarray(reference['scale'])
    centers=np.stack([np.median(transformed[keep][train.y[keep]==k],axis=0) for k in range(17)])
    assert np.isfinite(centers).all()
    w.json_write(root/'physical_reference.json',reference)
    w.atomic_npz(root/'training_prototypes.npz',centers=centers)
    if seed==42:
        candidates=list(Path(source).glob(f'full/*_subjects/S{subject:02d}/seed_42/oof/held_rep_6'))
        assert len(candidates)==1;fold=candidates[0]
        with np.load(fold/'input_scaler.npz') as a:
            np.testing.assert_array_equal(mean,a['mean']);np.testing.assert_array_equal(std,a['std'])
        for arm in w.EXPERTS:
            m=json.loads((fold/arm/'fit_manifest.json').read_text())
            assert m['training_window_sha256']==w.window_hash(train.meta)
            assert m['training_signal_sha256']==w.training_signal_hash(segments,train.meta)
    else:
        fold=root/'trained'
        config=dict(batch_size=512,dropout=.65,smoke=False)
        for arm in w.EXPERTS:
            model=w.train_fit(arm,seed,subject,train,fold/arm,'oof_held_6','db7-seven-matched-v1',config)
            del model;gc.collect();torch.cuda.empty_cache()
        w.atomic_npz(fold/'input_scaler.npz',mean=mean,std=std)
    hashes={arm:w.file_hash(fold/arm/'final_model.pt') for arm in w.EXPERTS}
    w.json_write(root/'NEURAL_LOCK.json',dict(subject=subject,seed=seed,train_repetitions=[1,3,4],calibration_repetitions=[6],test_repetitions=[2,5],checkpoint_sha256=hashes,scaler=prov,neural_fits=0 if seed==42 else 3,gpu=os.environ['CUDA_VISIBLE_DEVICES']))
    for split,reps in [('cal',[6]),('test',[2,5])]:
        table=w.fold_inventory(inventory,[1,3,4],reps)
        ds=w.make_windows(segments,table,mean,std,'evaluation')
        raw_eval=w.dataset_descriptors(ds)
        logits=np.empty((len(ds),3,17),np.float32)
        for j,arm in enumerate(w.EXPERTS):
            assert w.file_hash(fold/arm/'final_model.pt')==hashes[arm]
            model=AblationC1(arm,.65).cuda()
            model.load_state_dict(torch.load(fold/arm/'final_model.pt',map_location='cuda',weights_only=True))
            logits[:,j]=w.model_predict(model,ds,512)
            del model;gc.collect();torch.cuda.empty_cache()
        meta=ds.meta.copy();meta['seed']=seed
        meta.to_csv(root/f'{split}_metadata.csv',index=False)
        emg_level=np.mean((np.log(np.maximum(raw_eval[:,:12],reference['rms_floor']))-np.asarray(reference['center'])[:12])/np.asarray(reference['scale'])[:12],axis=1)
        w.atomic_npz(root/f'{split}.npz',logits=logits,y=ds.y,shifts=w.apply_shift_reference(raw_eval,reference),distances=distances(raw_eval,reference,centers),emg_level=emg_level,subject=np.full(len(ds),subject),rep=meta.native_repetition.to_numpy(),start=meta.window_start.to_numpy(),end=meta.window_end.to_numpy(),gesture=meta.gesture.to_numpy(),phase=meta.phase.to_numpy(dtype='U8'))
        del ds,raw_eval
    w.json_write(root/'completion.json',dict(success=True,subject=subject,seed=seed,checkpoint_sha256=hashes,exact_same_checkpoints_for_cal_and_test=True))
    print(f'CACHE COMPLETE S{subject:02d} seed{seed}',flush=True)

def worker(gpu,seeds,source,out):
    import torch
    assert torch.cuda.device_count()==1
    torch.set_num_threads(1)
    w.gpu_preflight(gpu,out)
    for seed in seeds:
        for subject in range(gpu+1,23,2):subject_cache(subject,seed,source,out)

def prepare_cache(out,seeds=(42,)):
    import torch
    assert torch.cuda.device_count()==2, 'Both T4 GPUs required for this stage'
    source=extract_checkpoints(source_archive(),Path('/kaggle/working/source_checkpoints')) if 42 in seeds else Path('/kaggle/working/not_used')
    jobs=[]
    for gpu in [0,1]:
        env=dict(os.environ,CUDA_VISIBLE_DEVICES=str(gpu),OMP_NUM_THREADS='1',OPENBLAS_NUM_THREADS='1')
        args=[sys.executable,__file__,'--gpu',str(gpu),'--seeds',','.join(map(str,seeds)),'--source',str(source),'--out',str(out)]
        jobs.append(subprocess.Popen(args,env=env))
    codes=[p.wait() for p in jobs]
    assert codes==[0,0],f'GPU workers failed: {codes}'
    for seed in seeds:
        for subject in range(1,23):
            done=json.loads((Path(out)/f'seed_{seed}/S{subject:02d}/completion.json').read_text())
            assert done['success'] and done['exact_same_checkpoints_for_cal_and_test']

if __name__=='__main__':
    ap=argparse.ArgumentParser();ap.add_argument('--gpu',type=int);ap.add_argument('--seeds');ap.add_argument('--source');ap.add_argument('--out');a=ap.parse_args()
    worker(a.gpu,list(map(int,a.seeds.split(','))),a.source,a.out)


## suite_study.py

Subject-grouped development selection, nested baseline cross-fitting, label-free gate features, selection locks, test metrics and study orchestration.

In [ ]:
%%writefile /kaggle/working/db7_suite_source/suite_study.py
"""Seven sequential, development-selected BRB studies on matched checkpoints.

Neural experts are subject-specific. The compact gate is shared across subjects.
Outer meta folds hold out SUBJECTS, not overlapping windows. Gate-fitting rows
use inner subject-cross-fitted temperature/alpha predictions. Test labels are
loaded only after the stage's selection lock has been written.
"""
from pathlib import Path
import argparse,copy,hashlib,json,zipfile
import numpy as np
import pandas as pd
from scipy.special import softmax
from scipy.stats import binomtest,wilcoxon
import brb_meta as old
import rule_core as rules

SUBJECTS=list(range(1,23));SEED=42
DEFAULT=dict(inputs='generic',target='correctness',counts=[2,2,2],temporal=None,kind='brb',learn_weights=False,enabled=True)
NAMES=['matched-checkpoints','directional-inputs','recovery-consequents','reference-capacity','causal-history','mechanism-controls','seed-robustness']

def write(path,obj):
    path=Path(path);path.parent.mkdir(parents=True,exist_ok=True)
    path.write_text(json.dumps(obj,indent=2,allow_nan=False),encoding='utf-8')

def sha(path):
    h=hashlib.sha256()
    with open(path,'rb') as f:
        for chunk in iter(lambda:f.read(8*1024*1024),b''):h.update(chunk)
    return h.hexdigest()

def extract(path,out):
    out=Path(out);out.mkdir(parents=True,exist_ok=True)
    with zipfile.ZipFile(path) as z:
        assert z.testzip() is None
        done=json.loads(z.read('completion.json'));assert done['success']
        for n in z.namelist():
            if n.endswith('/') :continue
            target=out/n;assert target.resolve().is_relative_to(out.resolve())
            target.parent.mkdir(parents=True,exist_ok=True);target.write_bytes(z.read(n))
    return out,done

def locate(stage):
    paths=list(Path('/kaggle/input').rglob(f'db7_{21+stage:03d}_results.zip'))
    assert len(paths)==1,f'Expected exactly one output archive for stage {stage}: {paths}'
    root,done=extract(paths[0],Path('/kaggle/working')/f'input_stage_{stage}')
    assert done['stage']==stage and done['subjects']==SUBJECTS
    return root,done

def load(cache,seed,split):
    bundles=[]
    for s in SUBJECTS:
        root=Path(cache)/f'seed_{seed}/S{s:02d}'
        done=json.loads((root/'completion.json').read_text());assert done['success']
        with np.load(root/f'{split}.npz',allow_pickle=False) as a:bundles.append({k:a[k] for k in a.files})
    result={k:np.concatenate([b[k] for b in bundles]) for k in bundles[0]}
    assert set(result['subject'])==set(SUBJECTS)
    assert set(result['rep'])==({6} if split=='cal' else {2,5})
    return result

def subset(d,idx):return {k:v[idx] for k,v in d.items()}
def groups(d):return d['subject']*10000+d['gesture']*100+d['rep']
def weights(d):return old._group_weights(groups(d))

def sample(d,cap=32):
    """Equal time-spread count per subject/gesture/trial, training only."""
    ids=[];g=groups(d)
    for key in np.unique(g):
        ix=np.flatnonzero(g==key);ix=ix[np.argsort(d['start'][ix],kind='stable')]
        ids.extend(ix[np.unique(np.linspace(0,len(ix)-1,min(cap,len(ix))).astype(int))])
    return np.asarray(ids,int)

def fit_base(d):
    assert set(d['rep'])=={6},'Only development repetition 6 may fit fusion'
    t,td=old.fit_temperatures(d['logits'],d['y'],groups(d))
    p=old._probabilities(d['logits'],t);alpha,ad=old._fit_alpha(p,d['y'],weights(d))
    return dict(temperatures=t,alpha=alpha,diagnostics=dict(temperature=td,alpha=ad),fit_subjects=np.unique(d['subject']).tolist())

def base_predict(model,d):
    p=old._probabilities(d['logits'],model['temperatures'])
    return p,np.einsum('njk,j->nk',p,np.asarray(model['alpha']))

def crossfit_base(d,nfold=3):
    p=np.zeros_like(d['logits'],dtype=float);base=np.zeros((len(p),17));audit=[]
    for held in np.array_split(np.unique(d['subject']),nfold):
        mask=np.isin(d['subject'],held);train=subset(d,~mask);model=fit_base(subset(train,sample(train)))
        assert not set(model['fit_subjects'])&set(held)
        p[mask],base[mask]=base_predict(model,subset(d,mask))
        audit.append(dict(fit_subjects=model['fit_subjects'],held_subjects=held.tolist()))
    return p,base,audit

def temporal(d,base):
    """Past observed outputs only, reset on timestamp gaps; no gesture/phase use."""
    result=np.zeros((len(base),3));result[:,1:]=.5
    pred=base.argmax(1);top=np.sort(base,axis=1);margin=top[:,-1]-top[:,-2]
    for s in np.unique(d['subject']):
        order=np.flatnonzero(d['subject']==s);order=order[np.argsort(d['start'][order],kind='stable')]
        history=[];last=None
        for i in order:
            if last is None or d['start'][i]-d['start'][last]!=20:history=[]
            if history:
                result[i,0]=np.mean(pred[history]==pred[i])
                result[i,1]=np.clip(.5+.5*(margin[i]-margin[history[0]]),0,1)
                result[i,2]=.5+.5*np.tanh((d['emg_level'][i]-d['emg_level'][history[0]])/2)
            history=(history+[i])[-10:];last=i
    return result

def features(d,p,base,config):
    b=base.argmax(1);scores=p.max(1).copy();scores[np.arange(len(b)),b]=-1
    c=scores.argmax(1);expert=p[np.arange(len(b)),:,c].argmax(1)
    r=np.arange(len(b));j=expert
    if config['target']=='correctness':
        c_all=p.argmax(2);js=np.tile(np.arange(3),(len(b),1));rr=r[:,None];bb=b[:,None]
    else:c_all=c[:,None];js=j[:,None];rr=r[:,None];bb=b[:,None]
    if config['inputs']=='generic':
        generic=old.indicators(p,d['shifts']);q=generic if config['target']=='correctness' else generic[r,j][:,None,:]
        names=['entropy','disagreement','global_deviation']
    else:
        bm=np.clip(base[rr,bb]-base[rr,c_all],0,1)
        cm=.5+.5*(p[rr,js,c_all]-p[rr,js,bb])
        db=d['distances'][rr,js,bb];dc=d['distances'][rr,js,c_all]
        contrast=.5+.5*(db-dc)/(db+dc+1e-9)
        q=np.stack([bm,cm,contrast],axis=-1);names=['baseline_margin','candidate_margin','prototype_contrast']
    if config.get('temporal'):
        k=['stability','margin_change','activation_change'].index(config['temporal'])
        tq=temporal(d,base)[:,k];q=np.concatenate([q,np.broadcast_to(tq[:,None,None],(*q.shape[:2],1))],axis=2)
        names.append(config['temporal'])
    assert np.isfinite(q).all() and np.all((q>=0)&(q<=1))
    return q,b,c,names

def train_gate(d,p,base,config):
    q,b,c,names=features(d,p,base,config);ix=sample(d);heads=[]
    if config['target']=='correctness':
        for j in range(3):
            y=(p[:,j].argmax(1)==d['y']).astype(int)
            heads.append(rules.fit(q[ix,j],y[ix],weights(subset(d,ix)),config['counts'],config['kind'],config['learn_weights'],k=2))
    else:
        # Classes: 0=recovery, 1=harm, 2=neither; b and c always differ.
        y=np.where(d['y']==c,0,np.where(d['y']==b,1,2))
        heads.append(rules.fit(q[ix,0],y[ix],weights(subset(d,ix)),config['counts'],config['kind'],config['learn_weights'],k=3))
    return dict(config=config,heads=heads,antecedents=names,fit_subjects=np.unique(d['subject']).tolist(),training_rows=len(ix),reference_fitting='training-fold median only')

def apply_gate(model,d,p,base):
    cfg=model['config'];q,b,c,_=features(d,p,base,cfg)
    if cfg['target']=='correctness':
        r=np.column_stack([rules.predict(head,q[:,j])[:,1] for j,head in enumerate(model['heads'])])
        # Match the baseline global weights using probabilities supplied by caller.
        alpha=np.asarray(model['alpha']);fusion=np.einsum('nj,njk->nk',r*alpha,p)/(r*alpha).sum(1)[:,None]
        return fusion.argmax(1),fusion,dict(reliability=r,candidate=c,baseline=b)
    outcome=rules.predict(model['heads'][0],q[:,0]);choose=outcome[:,0]>outcome[:,1]
    return np.where(choose,c,b),None,dict(outcome=outcome,candidate=c,baseline=b)

def accuracy_by_subject(d,pred):
    return np.array([(pred[d['subject']==s]==d['y'][d['subject']==s]).mean() for s in np.unique(d['subject'])])

def candidates(stage,previous):
    cfg=copy.deepcopy(previous or DEFAULT);cfg['enabled']=True
    if stage==1:return [dict(DEFAULT)]
    if stage==2:return [{**cfg,'inputs':x} for x in ['generic','directional']]
    if stage==3:return [{**cfg,'target':x} for x in ['correctness','recovery']]
    if stage==4:return [{**cfg,'counts':x} for x in [[2,2,2],[3,2,2],[3,3,2],[3,3,3]]]
    if stage==5:return [{**cfg,'temporal':x,'counts':cfg['counts']+([2] if x else [])} for x in [None,'stability','margin_change','activation_change']]
    if stage==6:return [{**cfg,'kind':k,'learn_weights':w} for k,w in [('brb',False),('brb',True),('sugeno',False),('logistic',False)]]
    return [cfg]

def select(d,options,out):
    assert set(d['rep'])=={6}
    rows=[];audit=[]
    for fold,held in enumerate(np.array_split(np.array(SUBJECTS),5)):
        mask=np.isin(d['subject'],held);tr=subset(d,~mask);val=subset(d,mask)
        inner_p,inner_b,inneraudit=crossfit_base(tr)
        baseline=fit_base(subset(tr,sample(tr)));p,b=base_predict(baseline,val)
        audit.append(dict(fold=fold,train_subjects=np.unique(tr['subject']).tolist(),validation_subjects=held.tolist(),inner=inneraudit))
        def record(name,pred):
            for s,acc in zip(held,accuracy_by_subject(val,pred)):rows.append(dict(fold=fold,subject=int(s),candidate=name,accuracy=float(acc)))
        record('baseline',b.argmax(1))
        confidence=p.max(2);cw=confidence*np.asarray(baseline['alpha']);cp=np.einsum('nj,njk->nk',cw,p)/cw.sum(1)[:,None]
        record('confidence',cp.argmax(1))
        for i,cfg in enumerate(options):
            gate=train_gate(tr,inner_p,inner_b,cfg);gate['alpha']=baseline['alpha']
            write(out/'development'/f'fold_{fold}_candidate_{i}.json',gate)
            pred,_,_=apply_gate(gate,val,p,b);record(str(i),pred)
        print(f'Meta fold {fold+1}/5 complete',flush=True)
    frame=pd.DataFrame(rows);frame.to_csv(out/'development_subject_scores.csv',index=False)
    summary=frame.groupby('candidate').accuracy.agg(['mean','std','count']).reset_index()
    summary.to_csv(out/'development_selection.csv',index=False);write(out/'fold_exclusion_audit.json',audit)
    means=frame.groupby('candidate').accuracy.mean()
    best=max(range(len(options)),key=lambda i:(means[str(i)],-np.prod(options[i]['counts']),-i))
    chosen=copy.deepcopy(options[best]);chosen['enabled']=bool(means[str(best)]>means['baseline']+1e-12)
    # Structural winner recorded even when neutral intervention wins.
    return chosen,dict(candidate=best,development_accuracy=float(means[str(best)]),baseline_accuracy=float(means['baseline']),criterion='mean subject accuracy; ties fewer rules then prespecified order',neutral_selected=not chosen['enabled'])

def metric(y,pred,p=None):
    cm=np.zeros((17,17),int);np.add.at(cm,(y,pred),1)
    result=dict(n=len(y),correct=int(np.sum(y==pred)),wrong=int(np.sum(y!=pred)),accuracy=float(np.mean(y==pred)),macro_f1=float(np.mean(2*cm.diagonal()/np.maximum(cm.sum(0)+cm.sum(1),1))),balanced_accuracy=float(np.mean(cm.diagonal()/np.maximum(cm.sum(1),1))))
    if p is not None:
        conf=p.max(1);ece=0.
        for lo in np.arange(15)/15:
            mask=(conf>=lo)&(conf<(lo+1/15) if lo<14/15 else conf<=1)
            if mask.any():ece+=mask.mean()*abs(conf[mask].mean()-(pred[mask]==y[mask]).mean())
        result.update(nll=float(-np.log(np.maximum(p[np.arange(len(y)),y],1e-12)).mean()),brier=float(((p-np.eye(17)[y])**2).sum(1).mean()),ece15=float(ece))
    return result,cm

def evaluate(d,base,pred,prob,out,seed,arm):
    folder=out/f'seed_{seed}'/arm;folder.mkdir(parents=True,exist_ok=True)
    oldpred=base.argmax(1);frame=pd.DataFrame({k:d[k] for k in ['subject','gesture','rep','start','end','phase']})
    frame['truth']=d['y']+1;frame['baseline']=oldpred+1;frame['prediction']=pred+1
    frame['correct']=pred==d['y'];frame['recovery']=(oldpred!=d['y'])&frame.correct;frame['harm']=(oldpred==d['y'])&~frame.correct
    frame.to_csv(folder/'predictions.csv.gz',index=False)
    if prob is not None:np.savez_compressed(folder/'probabilities.npz',probabilities=prob.astype(np.float32))
    rows=[]
    for s in SUBJECTS:
        ix=d['subject']==s;m,cm=metric(d['y'][ix],pred[ix],None if prob is None else prob[ix])
        m.update(subject=s,seed=seed,arm=arm,baseline_accuracy=float((oldpred[ix]==d['y'][ix]).mean()),recovery=int(frame.recovery[ix].sum()),harm=int(frame.harm[ix].sum()))
        rows.append(m);pd.DataFrame(cm,index=range(1,18),columns=range(1,18)).to_csv(folder/f'S{s:02d}_confusion.csv')
    metrics=pd.DataFrame(rows);metrics.to_csv(folder/'subject_metrics.csv',index=False)
    for keys,name in [(['subject','gesture'],'subject_gesture'),(['subject','gesture','rep'],'trial'),(['subject','phase'],'phase')]:
        grouped=frame.groupby(keys).agg(windows=('correct','size'),correct=('correct','sum'),accuracy=('correct','mean'),recovered=('recovery','sum'),harmed=('harm','sum'))
        grouped['wrong']=grouped.windows-grouped.correct;grouped.to_csv(folder/f'{name}_errors.csv')
    delta=(metrics.accuracy-metrics.baseline_accuracy).to_numpy();rng=np.random.default_rng(20260923)
    boot=np.mean(delta[rng.integers(0,22,size=(10000,22))],axis=1)
    signs=rng.choice([-1,1],size=(100000,22));permutation=(1+np.sum(np.abs((signs*delta).mean(1))>=abs(delta.mean())-1e-15))/100001
    recover=int(frame.recovery.sum());harm=int(frame.harm.sum())
    stats=dict(mean_subject_gain_pp=float(delta.mean()*100),subject_bootstrap_95ci_pp=(np.quantile(boot,[.025,.975])*100).tolist(),subject_signflip_p=float(permutation),wilcoxon_p=float(wilcoxon(delta).pvalue) if np.any(delta) else 1.,window_mcnemar_exact_p=float(binomtest(recover,recover+harm,.5).pvalue) if recover+harm else 1.,recovered=recover,harmed=harm,window_mcnemar_warning='Descriptive only: 95% overlapping windows violate independence; subject-cluster analysis is primary.',test_status='Reused DB7 test recordings; exploratory, not independent confirmation')
    write(folder/'significance.json',stats)
    return metrics

def fit_and_test(cache,seed,config,out):
    cal=load(cache,seed,'cal');p,b,audit=crossfit_base(cal)
    gate=train_gate(cal,p,b,config);baseline=fit_base(subset(cal,sample(cal)));gate['alpha']=baseline['alpha']
    write(out/f'seed_{seed}'/'fitted_gate.json',gate);write(out/f'seed_{seed}'/'fitted_baseline.json',baseline)
    write(out/f'seed_{seed}'/'final_crossfit_audit.json',audit)
    rule_rows=[]
    for j,h in enumerate(gate['heads']):
        consequences=['incorrect','correct'] if config['target']=='correctness' else ['recovery','harm','neither']
        for row in rules.rule_rows(h,gate['antecedents'],consequences):rule_rows.append(dict(head=j,**row))
    pd.DataFrame(rule_rows).to_csv(out/f'seed_{seed}'/'initial_and_trained_rules.csv',index=False)
    # This is deliberately the first load of test labels in the meta stage.
    assert (out/'selected_pipeline.json').exists()
    test=load(cache,seed,'test');p,base=base_predict(baseline,test);pred,prob,diag=apply_gate(gate,test,p,base)
    tables=[evaluate(test,base,base.argmax(1),base,out,seed,'baseline'),evaluate(test,base,pred,prob,out,seed,'structural_candidate')]
    selected=pred if config['enabled'] else base.argmax(1);selected_prob=prob if config['enabled'] else base
    tables.append(evaluate(test,base,selected,selected_prob,out,seed,'selected_policy'))
    for j,name in enumerate(['waveform','spectrum','inertial']):tables.append(evaluate(test,base,p[:,j].argmax(1),p[:,j],out,seed,name))
    confidence=p.max(2)*np.asarray(baseline['alpha']);cp=np.einsum('nj,njk->nk',confidence,p)/confidence.sum(1)[:,None]
    tables.append(evaluate(test,base,cp.argmax(1),cp,out,seed,'confidence'))
    np.savez_compressed(out/f'seed_{seed}'/'gate_diagnostics.npz',**diag)
    if config['target']=='recovery':
        y=np.where(test['y']==diag['candidate'],0,np.where(test['y']==diag['baseline'],1,2));op=diag['outcome']
        write(out/f'seed_{seed}'/'outcome_calibration.json',dict(nll=float(-np.log(np.maximum(op[np.arange(len(y)),y],1e-12)).mean()),brier=float(((op-np.eye(3)[y])**2).sum(1).mean()),consequents=['recovery','harm','neither'],multiclass_gesture_probability='Not defined for hard switching; no gesture NLL/Brier claimed'))
    return pd.concat(tables,ignore_index=True)

def main(stage):
    out=Path('/kaggle/working')/f'db7_{21+stage:03d}';out.mkdir(exist_ok=True)
    previous=None;provenance={}
    if stage==1:
        from suite_neural import prepare_cache
        cache=out/'cache';prepare_cache(cache,(42,))
    else:
        first,done=locate(1);cache=first/'cache'
        previous_root,previous_done=locate(stage-1) if stage>2 else (first,done)
        lock=previous_root/'selected_pipeline.json';previous=json.loads(lock.read_text())['config']
        provenance=dict(previous_stage=stage-1,previous_lock_sha256=sha(lock))
    cal=load(cache,42,'cal')
    if stage==7:
        chosen=previous;decision=dict(locked_from_stage=6,no_further_structure_selection=True)
    else:chosen,decision=select(cal,candidates(stage,previous),out)
    write(out/'selected_pipeline.json',dict(stage=stage,experiment_id=f'DB7-{21+stage:03d}',config=chosen,selection=decision,provenance=provenance,test_used_for_selection=False))
    tables=[fit_and_test(cache,42,chosen,out)]
    if stage==7:
        from suite_neural import prepare_cache
        newcache=out/'cache';prepare_cache(newcache,(43,44))
        for seed in [43,44]:tables.append(fit_and_test(newcache,seed,chosen,out))
    metrics=pd.concat(tables,ignore_index=True);metrics.to_csv(out/'all_subject_metrics.csv',index=False)
    # Holm correction across all reported nonbaseline arms within each seed.
    significance=[]
    for seed in sorted(metrics.seed.unique()):
        for path in sorted((out/f'seed_{seed}').glob('*/significance.json')):
            if path.parent.name=='baseline':continue
            record=json.loads(path.read_text());record.update(seed=int(seed),arm=path.parent.name);significance.append(record)
    sig=pd.DataFrame(significance)
    for seed,block in sig.groupby('seed'):
        order=block.sort_values('subject_signflip_p').index
        adjusted=np.maximum.accumulate(sig.loc[order,'subject_signflip_p'].to_numpy()*np.arange(len(order),0,-1))
        sig.loc[order,'subject_signflip_holm_p']=np.minimum(adjusted,1)
    sig.to_csv(out/'paired_significance_holm.csv',index=False)
    if stage==7:
        selected=metrics[metrics.arm=='selected_policy'].copy()
        selected['gain_pp']=100*(selected.accuracy-selected.baseline_accuracy)
        selected.groupby('subject').agg(mean_gain_pp=('gain_pp','mean'),min_gain_pp=('gain_pp','min'),max_gain_pp=('gain_pp','max')).to_csv(out/'seed_robustness_by_subject.csv')
    summary=metrics.groupby(['seed','arm']).agg(accuracy=('accuracy','mean'),macro_f1=('macro_f1','mean'),wrong=('wrong','sum'),recovery=('recovery','sum'),harm=('harm','sum'));summary.to_csv(out/'summary.csv')
    import matplotlib;matplotlib.use('Agg')
    import matplotlib.pyplot as plt
    pivot=metrics[metrics.arm.isin(['baseline','selected_policy'])].pivot_table(index='subject',columns='arm',values='accuracy')
    ax=pivot.plot.bar(figsize=(13,5));ax.set_ylabel('Accuracy (mean across available seeds)');ax.set_ylim(0,1);plt.tight_layout();plt.savefig(out/'subject_accuracy.png',dpi=160);plt.close()
    report=f'# DB7-{21+stage:03d}: {NAMES[stage-1]}\n\n'+summary.to_string()+'\n\nSelection: '+json.dumps(decision)+'\n\nNeural training: repetitions 1/3/4; development 6; test 2/5. Same checkpoint for calibration and test. Shared gate, subject-disjoint outer meta folds and inner cross-fitting. Neural models remain within-subject. 200 ms windows, 10 ms stride. Test results are exploratory on reused DB7 recordings. Seed robustness is not independent confirmation. No promised accuracy improvement. Initial and trained rules and all fit parameters are exported. Hard class-switch policies have no gesture probability metrics. Window McNemar is descriptive; subject clustering is primary.\n'
    (out/'REPORT.md').write_text(report)
    write(out/'completion.json',dict(success=True,stage=stage,subjects=SUBJECTS,seeds=[42,43,44] if stage==7 else [42],neural_fits=132 if stage==7 else 0,both_gpus_required=stage in [1,7],test_used_for_selection=False,source_sha256={p.name:sha(p) for p in Path(__file__).parent.glob('*.py')}))
    archive=out.parent/f'db7_{21+stage:03d}_results.zip'
    with zipfile.ZipFile(archive,'w',zipfile.ZIP_DEFLATED,compresslevel=2) as z:
        for path in out.rglob('*'):
            if path.is_file():z.write(path,path.relative_to(out))
    print(report,flush=True)

if __name__=='__main__':
    parser=argparse.ArgumentParser();parser.add_argument('--stage',type=int,choices=range(1,8),required=True);main(parser.parse_args().stage)


## test_suite.py

Scientific preflight: ER derivatives, probability normalization, convergence, causal invariance and refusal to fit fusion on test repetitions.

In [ ]:
%%writefile /kaggle/working/db7_suite_source/test_suite.py
"""Small scientific checks; synthetic data cannot establish research accuracy."""
import tempfile
from pathlib import Path
import numpy as np
from scipy.special import softmax
import rule_core as r
import suite_study as s

def run():
    # Execute the production directory setup through the loader call on a fresh
    # filesystem, without requiring CUDA or the DB7 recordings for this check.
    import ast
    from types import SimpleNamespace
    source=ast.parse(Path(__file__).with_name('suite_neural.py').read_text())
    fn=next(n for n in source.body if isinstance(n,ast.FunctionDef) and n.name=='subject_cache')
    setup=[]
    for node in fn.body:
        if isinstance(node,(ast.Import,ast.ImportFrom)):continue
        setup.append(node)
        if isinstance(node,ast.Assign) and isinstance(node.value,ast.Call) and isinstance(node.value.func,ast.Attribute) and node.value.func.attr=='load_subject':break
    def save_scaler(_input,subject,output):
        np.savez_compressed(output/'input_scaler.npz',mean=np.zeros(120),std=np.ones(120))
        return None,None,None,None
    from contextlib import nullcontext
    from uuid import uuid4
    with nullcontext(Path.cwd()/('directory_preflight_'+uuid4().hex)) as temporary:
        scope=dict(Path=Path,out=temporary,seed=42,subject=1,support=SimpleNamespace(find_input=lambda:None,load_subject=save_scaler))
        exec(compile(ast.Module(body=setup,type_ignores=[]),'directory_preflight','exec'),scope)
        with np.load(Path(temporary)/'seed_42/S01/inventory/input_scaler.npz') as saved:
            assert saved['mean'].shape==(120,)
    rng=np.random.default_rng(42);q=rng.random((40,3));refs=r.references(q,[3,3,3]);m,bits=r.matching(q,refs)
    a=np.prod(m,2);np.testing.assert_allclose(a.sum(1),1)
    beta=softmax(rng.normal(size=(27,3)),axis=1);g=rng.normal(size=(40,3))
    da,db=r.aggregate(a,beta,gradient=g);h=1e-6
    # Raw analytic ER derivatives include the normalization, tested off simplex.
    for i,j in [(0,0),(4,1),(20,2)]:
        plus=beta.copy();minus=beta.copy();plus[i,j]+=h;minus[i,j]-=h
        numeric=((r.aggregate(a,plus)-r.aggregate(a,minus))*g).sum()/(2*h)
        np.testing.assert_allclose(numeric,db[i,j],rtol=1e-5,atol=1e-6)
    for i,j in [(0,0),(4,1),(20,2)]:
        plus=a.copy();minus=a.copy();plus[i,j]+=h;minus[i,j]-=h
        numeric=((r.aggregate(plus,beta)-r.aggregate(minus,beta))*g).sum()/(2*h)
        np.testing.assert_allclose(numeric,da[i,j],rtol=1e-5,atol=1e-6)
    vertex=np.zeros((1,27));vertex[0,0]=1
    np.testing.assert_allclose(r.aggregate(vertex,beta)[0],beta[0])
    for kind,learn in [('brb',False),('brb',True),('sugeno',False),('logistic',False)]:
        model=r.fit(q,np.arange(40)%3,np.ones(40)/40,[2,2,2],kind,learn)
        p=r.predict(model,q);np.testing.assert_allclose(p.sum(1),1);assert np.isfinite(p).all()
        assert model['optimizer']['success'],model['optimizer']
    # Chronological features must be invariant to all future outputs/labels.
    d=dict(subject=np.ones(40),start=np.arange(40)*20,emg_level=rng.normal(size=40))
    p=softmax(rng.normal(size=(40,17)),axis=1);before=s.temporal(d,p)
    p[20:]=softmax(rng.normal(size=(20,17))*5,axis=1);d['emg_level'][20:]+=100
    np.testing.assert_array_equal(before[:20],s.temporal(d,p)[:20])
    # All antecedent/candidate modes are well shaped and do not need truth.
    d.update(logits=rng.normal(size=(40,3,17)),shifts=rng.random((40,3)),distances=rng.random((40,3,17)))
    experts=softmax(d['logits'],axis=2);base=experts.mean(1)
    for target in ['correctness','recovery']:
        for inputs in ['generic','directional']:
            cfg={**s.DEFAULT,'target':target,'inputs':inputs}
            features,b,c,_=s.features(d,experts,base,cfg)
            assert features.shape==(40,3 if target=='correctness' else 1,3)
            assert np.all(b!=c)
    bad={**d,'rep':np.full(40,2)}
    try:s.fit_base(bad)
    except AssertionError:pass
    else:raise AssertionError('Test repetition accepted for fitting')
    print('PASS: ER gradients, rule normalization, optimizer fits, label-free features, causal invariance, test fitting rejection')

if __name__=='__main__':run()


## Verify the mathematical and protocol preflight

Stop on failure; synthetic checks do not establish research accuracy.

In [ ]:
import subprocess,sys
subprocess.run([sys.executable,str(SOURCE/'test_suite.py')],check=True)


## Run this stage and export the evidence

Every later stage requires verified predecessor output. No fallback to an unrelated archive.

In [ ]:
subprocess.run([sys.executable,str(SOURCE/'suite_study.py'),'--stage','4'],check=True)


In [ ]:
from IPython.display import display,Markdown,Image
import pandas as pd
OUT=Path('/kaggle/working/db7_025')
display(Markdown((OUT/'REPORT.md').read_text()))
display(pd.read_csv(OUT/'summary.csv'))
display(Image(filename=str(OUT/'subject_accuracy.png')))
